# Phase 3 — RAG Retrieval Prototype

This notebook builds and evaluates the scientific retrieval layer.

Pipeline:

1. Load the validated scientific corpus
2. Remove empty pages
3. Clean extraction artifacts
4. Chunk scientific text
5. Generate stable chunk IDs
6. Preserve source/page metadata
7. Generate semantic embeddings
8. Store embeddings in ChromaDB
9. Implement top-k retrieval
10. Evaluate retrieval against golden cases

The retrieval system must achieve at least an 80% top-3 retrieval hit rate
before downstream LLM reasoning is added.

Our 5 PDFs
     ↓
Break them into sensible pieces
     ↓
Give every piece an ID
     ↓
Remember its PDF + page number
     ↓
MiniLM understands the meaning
     ↓
Store everything in ChromaDB
     ↓
Ask a question
     ↓
Find the 3 most relevant pieces
     ↓
Test using GC01–GC07
     ↓
At least 6/7 work correctly
     ↓
PHASE 3 COMPLETE ✅

## 1. Load and Prepare Scientific Corpus

The validated page-level scientific corpus created during Phase 2 is loaded
for retrieval preparation. Empty or non-extractable pages are excluded from
the retrieval corpus while the original processed corpus remains unchanged.

In [1]:
import pandas as pd

corpus_path = "../data/processed/scientific_corpus_pages.csv"

corpus_df = pd.read_csv(corpus_path)

print("Corpus loaded successfully.")
print("Shape:", corpus_df.shape)
print("\nColumns:")
print(corpus_df.columns.tolist())

corpus_df.head()

Corpus loaded successfully.
Shape: (1377, 4)

Columns:
['source_id', 'filename', 'page_number', 'text']


,source_id,filename,page_number,text
0,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,1,STATE of KNOWLEDGE\r\nof SOIL BIODIVERSITY\r\n...
1,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,2,NaN
2,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,3,STATE of KNOWLEDGE\r\nof SOIL BIODIVERSITY\r\n...
3,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,"Required citation\r\nFAO, ITPS, GSBI, CBD and ..."
4,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,5,III\r\nCONTENTS\r\nContributors\r\nXVI\r\nFor...


In [2]:
print("Total page records:", len(corpus_df))
print("Sources:", sorted(corpus_df["source_id"].unique()))

print("\nMissing values:")
print(corpus_df.isnull().sum())

Total page records: 1377
Sources: ['SRC01', 'SRC02', 'SRC03', 'SRC04', 'SRC05']

Missing values:
source_id       0
filename        0
page_number     0
text           25
dtype: int64


In [3]:
retrieval_df = corpus_df.copy()

# Remove pages with no extracted text
retrieval_df = retrieval_df.dropna(subset=["text"])

# Remove whitespace-only text
retrieval_df["text"] = retrieval_df["text"].astype(str).str.strip()
retrieval_df = retrieval_df[retrieval_df["text"] != ""].copy()

# Reset row numbers
retrieval_df.reset_index(drop=True, inplace=True)

In [4]:
print("Original pages:", len(corpus_df))
print("Retrievable pages:", len(retrieval_df))
print("Removed empty pages:", len(corpus_df) - len(retrieval_df))

print("\nPages remaining per source:")
print(retrieval_df.groupby("source_id").size())

print("\nRemaining null text:", retrieval_df["text"].isnull().sum())
print(
    "Remaining empty text:",
    (retrieval_df["text"].str.strip() == "").sum()
)

Original pages: 1377
Retrievable pages: 1352
Removed empty pages: 25

Pages remaining per source:
source_id
SRC01    613
SRC02    560
SRC03    166
SRC04      5
SRC05      8
dtype: int64

Remaining null text: 0
Remaining empty text: 0


## 2. Text Cleaning and Sentence-Aware Chunking

Extracted PDF text is cleaned conservatively to normalize whitespace while
preserving scientific values, units, citations, and terminology.

The cleaned text is then divided into sentence-aware chunks so that scientific
statements are not unnecessarily split in the middle of a sentence.

In [5]:
import re

def clean_text(text):
    # Replace line breaks/tabs/repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text)

    return text.strip()


retrieval_df["clean_text"] = retrieval_df["text"].apply(clean_text)

In [6]:
print("Empty cleaned text:",
      (retrieval_df["clean_text"] == "").sum())

print("\nExample BEFORE:\n")
print(retrieval_df.iloc[0]["text"][:500])

print("\nExample AFTER:\n")
print(retrieval_df.iloc[0]["clean_text"][:500])

Empty cleaned text: 0

Example BEFORE:

STATE of KNOWLEDGE
of SOIL BIODIVERSITY
Status, challenges and potentialities
Report
2020
﻿

Example AFTER:

STATE of KNOWLEDGE of SOIL BIODIVERSITY Status, challenges and potentialities Report 2020 ﻿


In [7]:
def split_into_sentences(text):
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

In [8]:
sample_text = retrieval_df.iloc[10]["clean_text"]

sample_sentences = split_into_sentences(sample_text)

print("Number of sentences:", len(sample_sentences))

for sentence in sample_sentences[:10]:
    print("\n-", sentence)

Number of sentences: 1

- X 7.1 | Assessment of soil biodiversit 430 7.1.1 | Contributions of soil biodiversity to ecosystems services 430 7.1.2 | National assessment 430 7.1.3 | Practical applications of soil biodiversit 431 7.1.4 | Major practices negatively impacting soil biodiversit 431 7.1.5 | Invasive alien species (IAS 431 7.1.6 | Monitoring soil biodiversit 432 7.1.7 | Indicators used to evaluate soil biodiversit 432 7.2 | Research, capacity development and awareness raisin 433 7.3 | Mainstreaming: policies, programmes, regulations and governmental framework 433 7.4 | Analysis of the main gaps, barriers and opportunities in the conservation and sustainable use of soil biodiversit 434 Annex II National Survey on Status of Soil Biodiversity: Knowledge, Challenges and Opportunitie 437 I | Assessmen 439 II | Research, capacity development and awareness raisin 442 III | Mainstreaming: policies, regulations and governmental framework 442 IV | Analysis of gaps and opportunities 443 An

In [9]:
sample_row = retrieval_df[
    (retrieval_df["source_id"] == "SRC01") &
    (retrieval_df["page_number"] == 100)
].iloc[0]

sample_text = sample_row["clean_text"]

print("Source:", sample_row["source_id"])
print("Page:", sample_row["page_number"])
print("\nTEXT SAMPLE:\n")
print(sample_text[:1500])

Source: SRC01
Page: 100

TEXT SAMPLE:

State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019). However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al., 2005; Milotić et al., 2018; Menéndez et al., 2016) and for sustaining multiple ecosystem functions (O’hea et al., 2010; Nervo et al., 2017; Manning et al., 2016; Piccini et al., 2018; Santos-Heredia et al., 2018) in both perturbed (Beynon et al., 2012) and unperturbed systems (Manning et al., 2017; Beynon et al., 2012). In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regulating the activity or co

In [10]:
sample_sentences = split_into_sentences(sample_text)

print("Number of sentences:", len(sample_sentences))

for i, sentence in enumerate(sample_sentences[:10], start=1):
    print(f"\nSentence {i}:")
    print(sentence)

Number of sentences: 13

Sentence 1:
State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019).

Sentence 2:
However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al., 2005; Milotić et al., 2018; Menéndez et al., 2016) and for sustaining multiple ecosystem functions (O’hea et al., 2010; Nervo et al., 2017; Manning et al., 2016; Piccini et al., 2018; Santos-Heredia et al., 2018) in both perturbed (Beynon et al., 2012) and unperturbed systems (Manning et al., 2017; Beynon et al., 2012).

Sentence 3:
In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regul

In [14]:
def create_chunks(text, target_size=1200, overlap_sentences=2):
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    # Fallback:
    # If sentence splitting produces one extremely long block
    # (common for TOCs, lists, or badly structured PDF text),
    # split it safely by character windows.
    if len(sentences) == 1 and len(sentences[0]) > target_size:
        chunks = []
        start = 0
        overlap_chars = 200

        while start < len(text):
            end = start + target_size
            chunk = text[start:end].strip()

            if chunk:
                chunks.append(chunk)

            if end >= len(text):
                break

            start = end - overlap_chars

        return chunks

    # Normal sentence-aware chunking
    chunks = []
    current_sentences = []

    for sentence in sentences:
        candidate = " ".join(current_sentences + [sentence])

        if len(candidate) <= target_size or not current_sentences:
            current_sentences.append(sentence)

        else:
            chunk = " ".join(current_sentences).strip()

            if chunk:
                chunks.append(chunk)

            current_sentences = (
                current_sentences[-overlap_sentences:]
                if overlap_sentences > 0
                else []
            )

            current_sentences.append(sentence)

    if current_sentences:
        chunk = " ".join(current_sentences).strip()

        if chunk:
            chunks.append(chunk)

    return chunks

In [13]:
sample_chunks = create_chunks(sample_text)

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks, start=1):
    print(f"\nCHUNK {i} — {len(chunk)} characters")
    print(chunk[:250])

Number of chunks: 6

CHUNK 1 — 1001 characters
State of knowledge of soil biodiversity 70 beetle abundance may be more important for ecosystem functioning than species richness (Manning and Cutler, 2018; Alvarado et al., 2019). However, functional group richness, species composition and maintenan

CHUNK 2 — 1073 characters
However, functional group richness, species composition and maintenance of interactions among co-occurring species (such as tunnelling and dwelling dung beetles; Nervo et al., 2017) have been shown to be important for ecosystem function (Larsen et al

CHUNK 3 — 1060 characters
In addition to their direct consumptive and shredding activities, macrofauna can indirectly affect soil C dynamics by altering the abundance and activity of their prey and/ or regulating the activity or composition of soil microbial communities. For 

CHUNK 4 — 1162 characters
Larger invertebrate predators, such as predatory beetles and spiders, can also have cascading effects through the food 

In [15]:
normal_chunks = create_chunks(sample_text)

print("Normal scientific page")
print("Number of chunks:", len(normal_chunks))

for i, chunk in enumerate(normal_chunks, 1):
    print(f"Chunk {i}: {len(chunk)} characters")

Normal scientific page
Number of chunks: 6
Chunk 1: 1001 characters
Chunk 2: 1073 characters
Chunk 3: 1060 characters
Chunk 4: 1162 characters
Chunk 5: 1196 characters
Chunk 6: 974 characters


In [16]:
toc_text = retrieval_df.iloc[10]["clean_text"]

print("Sentences detected:", len(split_into_sentences(toc_text)))

toc_chunks = create_chunks(toc_text)

print("TOC chunks:", len(toc_chunks))

for i, chunk in enumerate(toc_chunks, start=1):
    print(f"Chunk {i}: {len(chunk)} characters")

Sentences detected: 1
TOC chunks: 1
Chunk 1: 1081 characters


In [ ]:
## 3. Generate Corpus Chunks and Metadata

The validated retrieval corpus is divided into manageable chunks. Each chunk
receives a deterministic identifier and retains its original source, filename,
page number, and within-page chunk index for provenance and citation tracing.

In [17]:
all_chunks = []

for _, row in retrieval_df.iterrows():

    page_chunks = create_chunks(row["clean_text"])

    for chunk_index, chunk in enumerate(page_chunks, start=1):

        chunk_id = (
            f'{row["source_id"]}_'
            f'P{int(row["page_number"]):04d}_'
            f'C{chunk_index:03d}'
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk
        })

In [18]:
chunks_df = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_df))
print("Columns:", chunks_df.columns.tolist())

chunks_df.head()

Total chunks: 5652
Columns: ['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text']


,chunk_id,source_id,filename,page_number,chunk_index,text
0,SRC01_P0001_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,1,1,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
1,SRC01_P0003_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,3,1,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
2,SRC01_P0004_C001,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,1,"Required citation FAO, ITPS, GSBI, CBD and EC...."
3,SRC01_P0004_C002,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,2,The views expressed in this information produc...
4,SRC01_P0004_C003,SRC01,SRC01_State of Knowledge of Soil Biodiversity.pdf,4,3,"If a translation of this work is created, it m..."


In [19]:
print("Total chunks:", len(chunks_df))

print("Duplicate chunk IDs:",
      chunks_df["chunk_id"].duplicated().sum())

print("Missing chunk IDs:",
      chunks_df["chunk_id"].isnull().sum())

print("Missing text:",
      chunks_df["text"].isnull().sum())

print("Empty chunks:",
      (chunks_df["text"].str.strip() == "").sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

print("\nChunk length statistics:")
print(chunks_df["text"].str.len().describe())

Total chunks: 5652
Duplicate chunk IDs: 0
Missing chunk IDs: 0
Missing text: 0
Empty chunks: 0

Chunks per source:
source_id
SRC01    1985
SRC02    3003
SRC03     573
SRC04      18
SRC05      73
dtype: int64

Chunk length statistics:
count    5652.000000
mean     1062.437721
std       314.554712
min         3.000000
25%       970.000000
50%      1100.000000
75%      1168.000000
max      4828.000000
Name: text, dtype: float64


In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully.")
print("Maximum sequence length:", embedding_model.max_seq_length)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\shubham chavan\OneDrive\Documents\Desktop\biodiversity-intelligence-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shubham chavan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Maximum sequence length: 256


In [21]:
tokenizer = embedding_model.tokenizer

def get_token_count(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )
    return len(encoded["input_ids"])

chunks_df["token_count"] = chunks_df["text"].apply(get_token_count)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (269 > 256). Running this sequence through the model will result in indexing errors


In [22]:
print("Token count statistics:")
print(chunks_df["token_count"].describe())

too_long = chunks_df[
    chunks_df["token_count"] > embedding_model.max_seq_length
]

print("\nChunks exceeding 256 tokens:", len(too_long))

print(
    "Percentage exceeding limit:",
    round(len(too_long) / len(chunks_df) * 100, 2),
    "%"
)

print("Maximum token count:", chunks_df["token_count"].max())

Token count statistics:
count    5652.000000
mean      234.959837
std        85.767875
min         3.000000
25%       192.000000
50%       223.000000
75%       259.000000
max       901.000000
Name: token_count, dtype: float64

Chunks exceeding 256 tokens: 1462
Percentage exceeding limit: 25.87 %
Maximum token count: 901


In [23]:
tiny_chunks = chunks_df[
    chunks_df["token_count"] < 20
]

print("Chunks under 20 tokens:", len(tiny_chunks))

tiny_chunks[
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
].head(20)

Chunks under 20 tokens: 33


,chunk_id,source_id,page_number,token_count,text
0,SRC01_P0001_C001,SRC01,1,16,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
40,SRC01_P0022_C001,SRC01,22,5,© Andy Murray
57,SRC01_P0030_C001,SRC01,30,5,© Andy Murray
69,SRC01_P0035_C001,SRC01,35,5,© Andy Murray
70,SRC01_P0036_C001,SRC01,36,12,©FAO / Matteo Sala © Andy Murray
278,SRC01_P0121_C001,SRC01,121,15,© Pavel Krásenský Global diversity and distrib...
360,SRC01_P0143_C001,SRC01,143,5,© Andy Murray
361,SRC01_P0144_C001,SRC01,144,5,© Andy Murray
393,SRC01_P0156_C001,SRC01,156,9,©FAO / Matteo Sala
611,SRC01_P0206_C001,SRC01,206,12,State of knowledge of soil biodiversity 176 © ...


In [24]:
def create_token_aware_chunks(
    text,
    tokenizer,
    target_tokens=220,
    overlap_sentences=2
):
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    def token_count(value):
        return len(
            tokenizer(
                value,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

    # --------------------------------------------------
    # Handle a single sentence/block that is too long
    # --------------------------------------------------
    expanded_sentences = []

    for sentence in sentences:

        if token_count(sentence) <= target_tokens:
            expanded_sentences.append(sentence)
            continue

        # Token-level fallback for oversized sentences,
        # TOCs, lists or badly extracted PDF blocks
        token_ids = tokenizer(
            sentence,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        # Leave room for special tokens
        safe_size = target_tokens - 2

        for start in range(0, len(token_ids), safe_size):

            token_piece = token_ids[start:start + safe_size]

            piece = tokenizer.decode(
                token_piece,
                skip_special_tokens=True
            ).strip()

            if piece:
                expanded_sentences.append(piece)

    # --------------------------------------------------
    # Pack sentences into token-safe chunks
    # --------------------------------------------------
    chunks = []
    current_sentences = []

    for sentence in expanded_sentences:

        candidate = " ".join(
            current_sentences + [sentence]
        ).strip()

        if (
            not current_sentences
            or token_count(candidate) <= target_tokens
        ):
            current_sentences.append(sentence)

        else:
            chunk = " ".join(current_sentences).strip()

            if chunk:
                chunks.append(chunk)

            # Keep previous sentences for context
            overlap = (
                current_sentences[-overlap_sentences:]
                if overlap_sentences > 0
                else []
            )

            # Make sure overlap itself doesn't make
            # the next chunk exceed the token target
            while overlap:
                candidate = " ".join(
                    overlap + [sentence]
                ).strip()

                if token_count(candidate) <= target_tokens:
                    break

                overlap = overlap[1:]

            current_sentences = overlap + [sentence]

    if current_sentences:
        final_chunk = " ".join(current_sentences).strip()

        if final_chunk:
            chunks.append(final_chunk)

    return chunks

In [25]:
test_chunks = create_token_aware_chunks(
    sample_text,
    tokenizer
)

print("Chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks, 1):

    tokens = len(
        tokenizer(
            chunk,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    print(
        f"Chunk {i}: "
        f"{len(chunk)} characters | "
        f"{tokens} tokens"
    )

Chunks: 7
Chunk 1: 755 characters | 197 tokens
Chunk 2: 820 characters | 206 tokens
Chunk 3: 1060 characters | 207 tokens
Chunk 4: 800 characters | 169 tokens
Chunk 5: 798 characters | 201 tokens
Chunk 6: 759 characters | 175 tokens
Chunk 7: 974 characters | 204 tokens


In [30]:
chunks_df["token_count"] = chunks_df["text"].apply(
    lambda text: len(
        tokenizer(
            text,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )
)

print("Token counts calculated successfully.")
print(chunks_df.columns.tolist())

Token counts calculated successfully.
['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text', 'token_count']


In [31]:
all_chunks = []

for _, row in retrieval_df.iterrows():

    page_chunks = create_token_aware_chunks(
        row["clean_text"],
        tokenizer,
        target_tokens=220,
        overlap_sentences=2
    )

    for chunk_index, chunk in enumerate(page_chunks, start=1):

        chunk_id = (
            f'{row["source_id"]}_'
            f'P{int(row["page_number"]):04d}_'
            f'C{chunk_index:03d}'
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)

print("New total chunks:", len(chunks_df))

New total chunks: 6876


In [34]:
print("Total chunks:", len(chunks_df))

print("\nIntegrity checks:")
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())
print("Missing IDs:", chunks_df["chunk_id"].isnull().sum())
print("Missing text:", chunks_df["text"].isnull().sum())
print("Empty chunks:", (chunks_df["text"].str.strip() == "").sum())

print("\nToken statistics:")
print(chunks_df["token_count"].describe())

print("\nChunks > 220:",
      (chunks_df["token_count"] > 220).sum())

print("Chunks > 256:",
      (chunks_df["token_count"] > 256).sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

Total chunks: 6876

Integrity checks:
Duplicate IDs: 0
Missing IDs: 0
Missing text: 0
Empty chunks: 0

Token statistics:


KeyError: 'token_count'

In [35]:
print(chunks_df.columns.tolist())

['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text']


In [36]:
def count_tokens(text):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

chunks_df["token_count"] = chunks_df["text"].apply(count_tokens)

print("Done!")
print(chunks_df.columns.tolist())

Done!
['chunk_id', 'source_id', 'filename', 'page_number', 'chunk_index', 'text', 'token_count']


In [37]:
chunks_df[["chunk_id", "token_count"]].head()

,chunk_id,token_count
0,SRC01_P0001_C001,16
1,SRC01_P0003_C001,27
2,SRC01_P0004_C001,208
3,SRC01_P0004_C002,209
4,SRC01_P0004_C003,187


In [38]:
print("Total chunks:", len(chunks_df))
print("Maximum tokens:", chunks_df["token_count"].max())

print(
    "Chunks > 220:",
    (chunks_df["token_count"] > 220).sum()
)

print(
    "Chunks > 256:",
    (chunks_df["token_count"] > 256).sum()
)

print("\nToken statistics:")
print(chunks_df["token_count"].describe())

Total chunks: 6876
Maximum tokens: 223
Chunks > 220: 8
Chunks > 256: 0

Token statistics:
count    6876.000000
mean      185.630745
std        39.050009
min         3.000000
25%       176.000000
50%       199.000000
75%       212.000000
max       223.000000
Name: token_count, dtype: float64


In [39]:
credit_mask = (
    chunks_df["text"].str.contains(
        r"^\s*©",
        regex=True,
        na=False
    )
    &
    (chunks_df["token_count"] < 20)
)

credit_chunks = chunks_df[credit_mask]

print("Obvious credit/noise chunks:", len(credit_chunks))

credit_chunks[
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
]

Obvious credit/noise chunks: 18


,chunk_id,source_id,page_number,token_count,text
49,SRC01_P0022_C001,SRC01,22,5,© Andy Murray
71,SRC01_P0030_C001,SRC01,30,5,© Andy Murray
85,SRC01_P0035_C001,SRC01,35,5,© Andy Murray
86,SRC01_P0036_C001,SRC01,36,12,©FAO / Matteo Sala © Andy Murray
352,SRC01_P0121_C001,SRC01,121,15,© Pavel Krásenský Global diversity and distrib...
453,SRC01_P0143_C001,SRC01,143,5,© Andy Murray
454,SRC01_P0144_C001,SRC01,144,5,© Andy Murray
495,SRC01_P0156_C001,SRC01,156,9,©FAO / Matteo Sala
796,SRC01_P0220_C001,SRC01,220,8,©FAO/Ronald Vargas
878,SRC01_P0239_C001,SRC01,239,9,©FAO/Matteo Sala


In [40]:
before_count = len(chunks_df)

chunks_df = chunks_df[~credit_mask].copy()
chunks_df.reset_index(drop=True, inplace=True)

after_count = len(chunks_df)

print("Chunks before cleanup:", before_count)
print("Chunks after cleanup:", after_count)
print("Noise chunks removed:", before_count - after_count)

Chunks before cleanup: 6876
Chunks after cleanup: 6858
Noise chunks removed: 18


In [41]:
print("Final chunks:", len(chunks_df))

print("Duplicate IDs:",
      chunks_df["chunk_id"].duplicated().sum())

print("Missing text:",
      chunks_df["text"].isnull().sum())

print("Empty text:",
      (chunks_df["text"].str.strip() == "").sum())

print("Chunks > 256 tokens:",
      (chunks_df["token_count"] > 256).sum())

print("\nChunks per source:")
print(chunks_df.groupby("source_id").size())

Final chunks: 6858
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Chunks > 256 tokens: 0

Chunks per source:
source_id
SRC01    2620
SRC02    3526
SRC03     589
SRC04      25
SRC05      98
dtype: int64


In [42]:
weird_short = chunks_df[
    chunks_df["token_count"] < 20
][
    ["chunk_id", "source_id", "page_number", "token_count", "text"]
]

weird_short

,chunk_id,source_id,page_number,token_count,text
0,SRC01_P0001_C001,SRC01,1,16,STATE of KNOWLEDGE of SOIL BIODIVERSITY Status...
10,SRC01_P0005_C002,SRC01,5,17,soil biodiversity 98 2. 4. 1 | spatial pattern...
24,SRC01_P0012_C002,SRC01,12,11,responded to the survey 445 references 447
62,SRC01_P0027_C004,SRC01,27,15,of threatened species ld | land degradation ld...
66,SRC01_P0028_C004,SRC01,28,14,##cados pncti | technology and innovation plan
745,SRC01_P0206_C001,SRC01,206,12,State of knowledge of soil biodiversity 176 © ...
746,SRC01_P0207_C001,SRC01,207,12,Contributions of soil biodiversity to ecosyste...
747,SRC01_P0208_C001,SRC01,208,12,State of knowledge of soil biodiversity 178 © ...
748,SRC01_P0209_C001,SRC01,209,12,Contributions of soil biodiversity to ecosyste...
1035,SRC01_P0273_C004,SRC01,273,19,the highest rate of global warming and expansi...


### Chunk Quality Control

Obvious standalone copyright and image-credit fragments were removed from the
retrieval corpus. Short chunks were otherwise retained because some represent
meaningful section headings, captions, or scientific statements. Aggressive
length-based filtering was avoided to prevent accidental loss of valid
scientific context.

In [43]:
print("FINAL CHUNK CORPUS")
print("------------------")
print("Total chunks:", len(chunks_df))
print("Sources:", chunks_df["source_id"].nunique())
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())
print("Missing text:", chunks_df["text"].isnull().sum())
print("Empty text:", (chunks_df["text"].str.strip() == "").sum())
print("Maximum tokens:", chunks_df["token_count"].max())
print("Chunks exceeding MiniLM limit:",
      (chunks_df["token_count"] > embedding_model.max_seq_length).sum())

FINAL CHUNK CORPUS
------------------
Total chunks: 6858
Sources: 5
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Maximum tokens: 223
Chunks exceeding MiniLM limit: 0


## 4. Generate Semantic Embeddings

Each validated scientific chunk is converted into a semantic vector using
the `all-MiniLM-L6-v2` sentence-transformer model.

These embeddings represent the meaning of each chunk and will allow
scientifically relevant passages to be retrieved based on semantic similarity
rather than exact keyword matching.

In [44]:
test_texts = chunks_df["text"].head(3).tolist()

test_embeddings = embedding_model.encode(
    test_texts,
    normalize_embeddings=True
)

print("Number of test embeddings:", len(test_embeddings))
print("Embedding dimensions:", test_embeddings.shape)

Number of test embeddings: 3
Embedding dimensions: (3, 384)


In [45]:
all_embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding generation complete.")
print("Embedding matrix shape:", all_embeddings.shape)

Batches:   0%|          | 0/215 [00:00<?, ?it/s]

Embedding generation complete.
Embedding matrix shape: (6858, 384)


In [46]:
import numpy as np

print("Number of chunks:", len(chunks_df))
print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape[1])

print("Contains NaN:", np.isnan(all_embeddings).any())
print("Contains infinity:", np.isinf(all_embeddings).any())

Number of chunks: 6858
Number of embeddings: 6858
Embedding dimensions: 384
Contains NaN: False
Contains infinity: False


In [ ]:
## 5. Store Embeddings in ChromaDB

The validated scientific chunks and their MiniLM embeddings are stored in a
persistent ChromaDB collection.

Each vector retains its stable chunk identifier and provenance metadata,
including source document, filename, page number, and chunk index.

In [47]:
import chromadb

chroma_path = "../chroma_db"

client = chromadb.PersistentClient(
    path=chroma_path
)

print("ChromaDB client created.")

ChromaDB client created.


In [48]:
collection_name = "biodiversity_scientific_knowledge"

try:
    client.delete_collection(collection_name)
    print("Old collection removed.")
except Exception:
    print("No existing collection found.")

collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "Scientific biodiversity knowledge corpus",
        "embedding_model": "all-MiniLM-L6-v2",
        "chunking": "sentence-aware token-safe",
        "target_tokens": 220
    }
)

print("Fresh collection created.")

No existing collection found.
Fresh collection created.


In [49]:
ids = chunks_df["chunk_id"].tolist()
documents = chunks_df["text"].tolist()

metadatas = chunks_df[
    [
        "source_id",
        "filename",
        "page_number",
        "chunk_index"
    ]
].to_dict("records")

print("IDs:", len(ids))
print("Documents:", len(documents))
print("Metadata records:", len(metadatas))
print("Embeddings:", len(all_embeddings))

IDs: 6858
Documents: 6858
Metadata records: 6858
Embeddings: 6858


In [50]:
batch_size = 500

for start in range(0, len(chunks_df), batch_size):

    end = min(start + batch_size, len(chunks_df))

    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
        embeddings=all_embeddings[start:end].tolist()
    )

    print(f"Stored {end}/{len(chunks_df)} chunks")

Stored 500/6858 chunks
Stored 1000/6858 chunks
Stored 1500/6858 chunks
Stored 2000/6858 chunks
Stored 2500/6858 chunks
Stored 3000/6858 chunks
Stored 3500/6858 chunks
Stored 4000/6858 chunks
Stored 4500/6858 chunks
Stored 5000/6858 chunks
Stored 5500/6858 chunks
Stored 6000/6858 chunks
Stored 6500/6858 chunks
Stored 6858/6858 chunks


In [51]:
print("Chunks in DataFrame:", len(chunks_df))
print("Chunks in ChromaDB:", collection.count())

assert collection.count() == len(chunks_df)

print("\nChromaDB validation passed.")

Chunks in DataFrame: 6858
Chunks in ChromaDB: 6858

ChromaDB validation passed.


## 6. Semantic Top-k Retrieval

A semantic retrieval function is implemented to convert a user query into a
MiniLM embedding and search ChromaDB for the most relevant scientific chunks.

The retriever returns the top-k matching chunks together with their stable
chunk IDs, source documents, page numbers, and similarity distances for
traceable scientific evidence retrieval.

In [52]:
def retrieve(query, top_k=3):

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(len(results["ids"][0])):

        retrieved_chunks.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks

In [53]:
query = "How does agricultural diversification affect biodiversity?"

results = retrieve(query, top_k=3)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("Distance:", round(result["distance"], 4))

    print("\nText:")
    print(result["text"])


RESULT 1
Chunk ID: SRC05_P0002_C010
Source: SRC05
Page: 2
Distance: 0.4891

Text:
The most examined diversification practices were organic amendment, re- duced tillage, and crop diversification (146, 118, and 111 effect sizes), whereas noncrop diversification and inoculation, as well as organic farming, were less represented (38, 9, and 34 effect sizes) and need further investigations. Biodiversity and ecosystem service response to diversification The second-order meta-analysis showed that agricultural diversifi- cation strengthens several ecosystem service categories (omnibus test QM = 43.67; P < 0.0001; Fig. 2A) while having a neutral effect on crop yield [lnRR = 0.01; 95% confidence interval (CI) = −0.12 to 0.14].

RESULT 2
Chunk ID: SRC05_P0001_C008
Source: SRC05
Page: 1
Distance: 0.5092

Text:
Functional diversity below ground can also be supported and stimulated through addition of organic inputs (e.g., manure and crop residues) or reducing soil disturbance (e.g., reduced tillag

In [54]:
import pandas as pd

golden_df = pd.read_csv("../data/evaluation/golden_cases.csv")

golden_df

,case_id,question,known_soc,known_rainfall,known_land_use,known_biodiversity,missing_variables,required_knowledge,expected_behavior,forbidden_behavior
0,GC01,"A farm has low soil organic carbon, low rainfa...",0.3%,low,monoculture wheat,low species count,none,SOC <-> soil/ecosystem condition; SOC <-> biod...,Notice SOC + rainfall + land use + biodiversit...,Recommend generic 'plant more trees' without a...
1,GC02,"My land already has good soil carbon, decent r...",1.8%,adequate,agroforestry,high species count,none,Maintenance-level agroforestry/high-biodiversi...,Recognize this is a positive baseline; suggest...,Recommend a drastic intervention when conditio...
2,GC03,Biodiversity is declining on my land. Soil org...,0.4%,unknown,unknown,declining,"rainfall, land use",SOC <-> biodiversity decline indicators; what ...,Ask a targeted clarifying question for rainfal...,Fabricate rainfall or land use to produce an a...
3,GC04,We get decent rainfall but our soil carbon is ...,low,adequate,monoculture wheat,unknown,biodiversity indicator,SOC <-> monoculture interaction effects; how a...,Reason about SOC + land use with rainfall as a...,Assume biodiversity is low without evidence or...
4,GC05,What can I do to improve biodiversity on this ...,unknown,unknown,monoculture wheat,unknown,"soil organic carbon, rainfall, biodiversity in...",General monoculture <-> biodiversity evidence ...,Ask for the missing core variables (at minimum...,"Provide a fully specific, quantified recommend..."
5,GC06,"Rainfall here is high, but soil carbon is very...",very low,high,monoculture corn,moderate/stable,none,How high/non-limiting rainfall expands the ran...,Recognize that high rainfall removes the water...,Ignore rainfall as a relevant factor just beca...
6,GC07,"Semi-arid region, soil organic carbon 0.3%, ra...",0.3%,low,monoculture wheat,unknown,biodiversity indicator (optional),SOC <-> soil/ecosystem condition; rainfall (lo...,Recommend an evidence-supported intervention a...,Invent a quantified estimate or timeframe when...
7,GC08,What's the best fertilizer brand for my lawn?,unknown,unknown,lawn,unknown,not applicable - out of scope,Scope boundary definition for the system; how ...,Recognize the question falls outside the syste...,Fabricate an environmental recommendation for ...


## 7. Golden-Case Retrieval Evaluation

Retrieval quality is evaluated using the predefined Phase 1 golden cases.

For GC01–GC07, a case is considered a top-3 retrieval hit when at least one
of the three retrieved chunks contains scientific evidence relevant to the
case's predefined `required_knowledge`.

The criteria are fixed before inspecting retrieval results to avoid
post-hoc relevance judgments.

GC08 is evaluated separately as an out-of-scope case and is not included
in the scientific top-3 hit-rate calculation.

In [55]:
eval_df = golden_df[
    ["case_id", "question", "required_knowledge"]
].copy()

eval_df

,case_id,question,required_knowledge
0,GC01,"A farm has low soil organic carbon, low rainfa...",SOC <-> soil/ecosystem condition; SOC <-> biod...
1,GC02,"My land already has good soil carbon, decent r...",Maintenance-level agroforestry/high-biodiversi...
2,GC03,Biodiversity is declining on my land. Soil org...,SOC <-> biodiversity decline indicators; what ...
3,GC04,We get decent rainfall but our soil carbon is ...,SOC <-> monoculture interaction effects; how a...
4,GC05,What can I do to improve biodiversity on this ...,General monoculture <-> biodiversity evidence ...
5,GC06,"Rainfall here is high, but soil carbon is very...",How high/non-limiting rainfall expands the ran...
6,GC07,"Semi-arid region, soil organic carbon 0.3%, ra...",SOC <-> soil/ecosystem condition; rainfall (lo...
7,GC08,What's the best fertilizer brand for my lawn?,Scope boundary definition for the system; how ...


In [56]:
scientific_eval_df = eval_df[
    eval_df["case_id"] != "GC08"
].copy()

for _, row in scientific_eval_df.iterrows():
    print("=" * 80)
    print(row["case_id"])
    print("QUESTION:")
    print(row["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(row["required_knowledge"])
    print()

GC01
QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

GC02
QUESTION:
My land already has good soil carbon, decent rainfall, and agroforestry. Species counts are high. Is there anything more I should do?

REQUIRED KNOWLEDGE:
Maintenance-level agroforestry/high-biodiversity system evidence; SOC <-> biodiversity stability under good conditions; guidance on when no major intervention is warranted

GC03
QUESTION:
Biodiversity is declining on my land. Soil organic carbon is 0.4%.

REQUIRED KNOWLEDGE:
SOC <-> biodiversity decline indicators; what rainfall data is needed and why; what land-use data is needed and why; clarifying-question triggers

In [57]:
golden_retrieval_results = []

for _, row in scientific_eval_df.iterrows():

    results = retrieve(
        row["question"],
        top_k=3
    )

    for rank, result in enumerate(results, start=1):

        golden_retrieval_results.append({
            "case_id": row["case_id"],
            "question": row["question"],
            "required_knowledge": row["required_knowledge"],
            "rank": rank,
            "chunk_id": result["chunk_id"],
            "source_id": result["metadata"]["source_id"],
            "page_number": result["metadata"]["page_number"],
            "distance": result["distance"],
            "text": result["text"]
        })

golden_results_df = pd.DataFrame(golden_retrieval_results)

print(
    "Golden cases tested:",
    golden_results_df["case_id"].nunique()
)

print(
    "Total retrieved chunks:",
    len(golden_results_df)
)

Golden cases tested: 7
Total retrieved chunks: 21


In [58]:
for case_id in scientific_eval_df["case_id"]:

    case_results = golden_results_df[
        golden_results_df["case_id"] == case_id
    ]

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    print("\nQUESTION:")
    print(case_results.iloc[0]["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(case_results.iloc[0]["required_knowledge"])

    for _, result in case_results.iterrows():

        print("\n" + "-" * 80)

        print(
            f'RANK {result["rank"]} | '
            f'{result["chunk_id"]} | '
            f'{result["source_id"]} | '
            f'Page {result["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print("\nTEXT:")
        print(result["text"])


GC01

QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

--------------------------------------------------------------------------------
RANK 1 | SRC01_P0270_C005 | SRC01 | Page 270 | Distance 0.7656

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nut

In [59]:
for case_id in ["GC04", "GC07"]:

    case_results = golden_results_df[
        golden_results_df["case_id"] == case_id
    ]

    print("\n", "=" * 80)
    print(case_id)

    for _, row in case_results.iterrows():
        print(
            row["rank"],
            row["chunk_id"],
            row["source_id"],
            row["page_number"],
            round(row["distance"], 4)
        )


GC04
1 SRC01_P0601_C005 SRC01 601 0.865
2 SRC01_P0270_C005 SRC01 270 0.8686
3 SRC01_P0588_C001 SRC01 588 0.8877

GC07
1 SRC01_P0344_C001 SRC01 344 0.5801
2 SRC01_P0469_C001 SRC01 469 0.5877
3 SRC01_P0270_C005 SRC01 270 0.6326


In [60]:


for case_id in ["GC04", "GC07"]:

    question = scientific_eval_df.loc[
        scientific_eval_df["case_id"] == case_id,
        "question"
    ].iloc[0]

    diagnostic_results = retrieve(question, top_k=10)

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    for rank, result in enumerate(diagnostic_results, start=1):

        print(
            f'\nRANK {rank} | '
            f'{result["chunk_id"]} | '
            f'{result["metadata"]["source_id"]} | '
            f'Page {result["metadata"]["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print(result["text"][:500])


GC04

RANK 1 | SRC01_P0601_C005 | SRC01 | Page 601 | Distance 0.8650
2015. Soil fertility decline at the base of rural poverty in sub-Saharan Africa. Nature Plants, 1: 15101. doi: 10.1038/nplants.2015.101 Vargas-Rojas, R., Cuevas-Corona, R., Yigini, Y., Tong, Y., Bazza, Z. & Wiese, L. 2019. Unlocking the Potential of Soil Organic Carbon: A Feasible Way Forward. Pages 373–395 in T. Ginzky, H. Dooley, E. Heuser, I.L. Kasimbazi, E. Markus, T. Qin, eds. International Yearbook of Soil Law and Policy, 2018, pp. 373-395. Cham, Switzerland. Springer. Vasenev, V. & Kuzyak

RANK 2 | SRC01_P0270_C005 | SRC01 | Page 270 | Distance 0.8686
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion

In [61]:
diagnostic_queries = [
    "How does low soil organic carbon affect soil biodiversity and soil health?",
    "What restoration practices are suitable under low rainfall or semi-arid conditions?",
    "How can crop diversification, intercropping, or agroforestry improve biodiversity in monoculture farming?"
]

for query in diagnostic_queries:

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    results = retrieve(query, top_k=3)

    for rank, result in enumerate(results, start=1):

        print(
            f'\nRANK {rank} | '
            f'{result["chunk_id"]} | '
            f'{result["metadata"]["source_id"]} | '
            f'Page {result["metadata"]["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print(result["text"][:600])


QUERY: How does low soil organic carbon affect soil biodiversity and soil health?

RANK 1 | SRC01_P0236_C003 | SRC01 | Page 236 | Distance 0.4231
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor exp

RANK 2 | SRC01_P0227_C001 | SRC01 | Page 227 | Distance 0.5185
Threats to soil biodiversity - global and regional trends 197 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 C C C C C C C C C C C Impacts on soil biodiversity Erosion and landslides drivers 

### Multi-Query Retrieval

Complex user questions may contain several environmental factors whose relevant
evidence occurs in different scientific sources. A single query embedding can
underrepresent some of these factors.

The retriever therefore decomposes complex questions into focused evidence
queries covering soil organic carbon, rainfall/climate constraints, land-use
management, and biodiversity where applicable. Results are merged and
deduplicated using their stable chunk IDs.

In [67]:
def decompose_query(question):
    q = question.lower()

    queries = {
        "original": question
    }

    if any(term in q for term in [
        "soil carbon",
        "soil organic carbon",
        "soc"
    ]):
        queries["soc"] = (
            "How does soil organic carbon affect soil biodiversity "
            "and soil health?"
        )

    if any(term in q for term in [
        "rainfall",
        "semi-arid",
        "semi arid",
        "dryland",
        "drought"
    ]):
        queries["rainfall"] = (
            "How do rainfall and water availability affect "
            "restoration and vegetation establishment?"
        )

    if any(term in q for term in [
        "monoculture",
        "wheat",
        "corn",
        "maize",
        "agroforestry"
    ]):
        queries["land_use"] = (
            "How do agricultural land-use diversification, "
            "intercropping and agroforestry affect biodiversity?"
        )

    if any(term in q for term in [
        "biodiversity",
        "species",
        "species count",
        "species diversity"
    ]):
        queries["biodiversity"] = (
            "What agricultural management practices support "
            "biodiversity and ecosystem health?"
        )

    return queries

In [63]:
for case_id in ["GC01", "GC04", "GC07"]:

    question = scientific_eval_df.loc[
        scientific_eval_df["case_id"] == case_id,
        "question"
    ].iloc[0]

    print("\n", case_id)

    for query in decompose_query(question):
        print(" -", query)


 GC01
 - A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?
 - How does soil organic carbon affect soil biodiversity and soil health?
 - How do rainfall and water availability affect restoration and vegetation establishment?
 - How do agricultural land-use diversification, intercropping and agroforestry affect biodiversity?
 - What agricultural management practices support biodiversity and ecosystem health?

 GC04
 - We get decent rainfall but our soil carbon is low and we grow only wheat. What should we do?
 - How does soil organic carbon affect soil biodiversity and soil health?
 - How do rainfall and water availability affect restoration and vegetation establishment?
 - How do agricultural land-use diversification, intercropping and agroforestry affect biodiversity?

 GC07
 - Semi-arid region, soil organic carbon 0.3%, rainfall is low, crop is monoculture wheat. What should I do to improve biod

In [64]:
def retrieve_multi_query(question, top_k=3, per_query_k=5, rrf_k=60):

    queries = decompose_query(question)

    fused = {}

    for query in queries:

        results = retrieve(query, top_k=per_query_k)

        for rank, result in enumerate(results, start=1):

            chunk_id = result["chunk_id"]

            # Reciprocal Rank Fusion score
            score = 1 / (rrf_k + rank)

            if chunk_id not in fused:
                fused[chunk_id] = {
                    "chunk_id": chunk_id,
                    "text": result["text"],
                    "metadata": result["metadata"],
                    "rrf_score": 0.0,
                    "best_distance": result["distance"],
                    "matched_queries": []
                }

            fused[chunk_id]["rrf_score"] += score

            fused[chunk_id]["best_distance"] = min(
                fused[chunk_id]["best_distance"],
                result["distance"]
            )

            fused[chunk_id]["matched_queries"].append(query)

    ranked_results = sorted(
        fused.values(),
        key=lambda x: (
            -x["rrf_score"],
            x["best_distance"]
        )
    )

    return ranked_results[:top_k]

In [66]:
gc07_question = scientific_eval_df.loc[
    scientific_eval_df["case_id"] == "GC07",
    "question"
].iloc[0]

results = retrieve_multi_query(
    gc07_question,
    top_k=3
)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("RRF Score:", round(result["rrf_score"], 6))
    print("Best Distance:", round(result["best_distance"], 4))
    print("Matched Queries:", len(result["matched_queries"]))

    print("\nText:")
    print(result["text"])


RESULT 1
Chunk ID: SRC01_P0236_C003
Source: SRC01
Page: 236
RRF Score: 0.016393
Best Distance: 0.4176
Matched Queries: 1

Text:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor explaining biodiversity was pH, and low pH soils tend to have higher carbon content. Several authors also highlight the importance of soil carbon quality in addition to quantity for below-ground diversity on a global scale (Crowther et al., 2019). For instance, Szoboszlay e

In [68]:
def retrieve_coverage_aware(question, top_k=3, per_aspect_k=3):
    queries = decompose_query(question)

    # We prefer specific evidence aspects over the original broad query
    aspect_priority = [
        "soc",
        "rainfall",
        "land_use",
        "biodiversity"
    ]

    selected = []
    selected_ids = set()

    # STEP 1:
    # Select the best unique result from each relevant aspect
    for aspect in aspect_priority:

        if aspect not in queries:
            continue

        results = retrieve(
            queries[aspect],
            top_k=per_aspect_k
        )

        for result in results:

            if result["chunk_id"] not in selected_ids:

                selected.append({
                    **result,
                    "retrieval_aspect": aspect
                })

                selected_ids.add(result["chunk_id"])
                break

        if len(selected) == top_k:
            return selected

    # STEP 2:
    # If fewer than top_k results were found,
    # use the original question to fill remaining slots
    if len(selected) < top_k:

        original_results = retrieve(
            queries["original"],
            top_k=per_aspect_k
        )

        for result in original_results:

            if result["chunk_id"] not in selected_ids:

                selected.append({
                    **result,
                    "retrieval_aspect": "original"
                })

                selected_ids.add(result["chunk_id"])

            if len(selected) == top_k:
                break

    return selected

In [69]:
gc07_question = scientific_eval_df.loc[
    scientific_eval_df["case_id"] == "GC07",
    "question"
].iloc[0]

results = retrieve_coverage_aware(
    gc07_question,
    top_k=3
)

for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 80)
    print(f"RESULT {rank}")
    print("=" * 80)

    print("Aspect:", result["retrieval_aspect"])
    print("Chunk ID:", result["chunk_id"])
    print("Source:", result["metadata"]["source_id"])
    print("Page:", result["metadata"]["page_number"])
    print("Distance:", round(result["distance"], 4))

    print("\nText:")
    print(result["text"])


RESULT 1
Aspect: soc
Chunk ID: SRC01_P0236_C003
Source: SRC01
Page: 236
Distance: 0.4176

Text:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly impact soil biodiversity. For instance, soil biodiversity was higher in agricultural soils than in carbon-rich northern forests (Griffiths et al., 2016), but the main factor explaining biodiversity was pH, and low pH soils tend to have higher carbon content. Several authors also highlight the importance of soil carbon quality in addition to quantity for below-ground diversity on a global scale (Crowther et al., 2019). For instance, Szoboszlay et al. (2017) found evidence of a

In [70]:
coverage_results = []

for _, row in scientific_eval_df.iterrows():

    results = retrieve_coverage_aware(
        row["question"],
        top_k=3
    )

    for rank, result in enumerate(results, start=1):

        coverage_results.append({
            "case_id": row["case_id"],
            "question": row["question"],
            "required_knowledge": row["required_knowledge"],
            "rank": rank,
            "aspect": result["retrieval_aspect"],
            "chunk_id": result["chunk_id"],
            "source_id": result["metadata"]["source_id"],
            "page_number": result["metadata"]["page_number"],
            "distance": result["distance"],
            "text": result["text"]
        })

coverage_results_df = pd.DataFrame(coverage_results)

print(
    "Golden cases tested:",
    coverage_results_df["case_id"].nunique()
)

print(
    "Total retrieved chunks:",
    len(coverage_results_df)
)

Golden cases tested: 7
Total retrieved chunks: 21


In [71]:
for case_id in scientific_eval_df["case_id"]:

    case_results = coverage_results_df[
        coverage_results_df["case_id"] == case_id
    ]

    print("\n" + "=" * 100)
    print(case_id)
    print("=" * 100)

    print("\nQUESTION:")
    print(case_results.iloc[0]["question"])

    print("\nREQUIRED KNOWLEDGE:")
    print(case_results.iloc[0]["required_knowledge"])

    for _, result in case_results.iterrows():

        print("\n" + "-" * 80)

        print(
            f'RANK {result["rank"]} | '
            f'Aspect: {result["aspect"]} | '
            f'{result["chunk_id"]} | '
            f'{result["source_id"]} | '
            f'Page {result["page_number"]} | '
            f'Distance {result["distance"]:.4f}'
        )

        print("\nTEXT:")
        print(result["text"])


GC01

QUESTION:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

REQUIRED KNOWLEDGE:
SOC <-> soil/ecosystem condition; SOC <-> biodiversity; rainfall <-> vegetation establishment; rainfall <-> restoration constraints; monoculture <-> biodiversity; land-use diversification interventions; evidence on expected effects

--------------------------------------------------------------------------------
RANK 1 | Aspect: soc | SRC01_P0236_C003 | SRC01 | Page 236 | Distance 0.4176

TEXT:
Nonetheless, the generally positive relationship between soil C stock and soil biodiversity suggests that soil carbon loss is a threat to soil biodiversity. In this regard, Orgiazzi et al. (2016a) identified SOC decline as a major threat to both soil microbial and fauna biodiversity. But the underlying causes may be different, as the main drivers of SOC loss, land use change and climate change (see below) also directly i

### Additional Retrieval Noise Analysis

Inspect structural PDF content that may harm semantic retrieval,
including bibliography entries, questionnaire material, contents fragments,
and other non-evidence text.

In [2]:
import pandas as pd

corpus_df = pd.read_csv(
    "../data/processed/scientific_corpus_pages.csv"
)

print("Corpus loaded:", corpus_df.shape)

Corpus loaded: (1377, 4)


In [3]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer

# ============================================================
# 1. LOAD CORPUS
# ============================================================

corpus_df = pd.read_csv(
    "../data/processed/scientific_corpus_pages.csv"
)

corpus_df["text"] = corpus_df["text"].fillna("").astype(str)

print("Corpus pages loaded:", len(corpus_df))


# ============================================================
# 2. LOAD MINILM TOKENIZER
# ============================================================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = embedding_model.tokenizer

MAX_CHUNK_TOKENS = 220


# ============================================================
# 3. SENTENCE SPLITTER
# ============================================================

def split_sentences(text):

    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        text
    )

    return [s.strip() for s in sentences if s.strip()]


# ============================================================
# 4. TOKEN COUNT
# ============================================================

def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )


# ============================================================
# 5. TOKEN-SAFE CHUNKING
# ============================================================

def create_chunks(text, max_tokens=MAX_CHUNK_TOKENS):

    sentences = split_sentences(text)

    if not sentences:
        return []

    chunks = []
    current = []

    for sentence in sentences:

        candidate = " ".join(current + [sentence])

        # Sentence fits into current chunk
        if count_tokens(candidate) <= max_tokens:
            current.append(sentence)
            continue

        # Save current chunk
        if current:
            chunks.append(" ".join(current))

        # Sentence itself is too large
        if count_tokens(sentence) > max_tokens:

            words = sentence.split()
            temp = []

            for word in words:

                candidate = " ".join(temp + [word])

                if count_tokens(candidate) <= max_tokens:
                    temp.append(word)

                else:
                    if temp:
                        chunks.append(" ".join(temp))

                    temp = [word]

            current = temp

        else:
            current = [sentence]

    if current:
        chunks.append(" ".join(current))

    return chunks


# ============================================================
# 6. CREATE CHUNK DATAFRAME
# ============================================================

records = []

for _, row in corpus_df.iterrows():

    text = row["text"].strip()

    if not text:
        continue

    page_chunks = create_chunks(text)

    for chunk_index, chunk_text in enumerate(
        page_chunks,
        start=1
    ):

        chunk_id = (
            f'{row["source_id"]}'
            f'_P{int(row["page_number"]):04d}'
            f'_C{chunk_index:03d}'
        )

        records.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk_text
        })


chunks_df = pd.DataFrame(records)


# ============================================================
# 7. TOKEN COUNTS
# ============================================================

chunks_df["token_count"] = chunks_df["text"].apply(
    count_tokens
)


# ============================================================
# 8. REMOVE VERY SHORT / OBVIOUS NON-CONTENT CHUNKS
# ============================================================

chunks_df = chunks_df[
    chunks_df["token_count"] >= 20
].copy()

chunks_df.reset_index(drop=True, inplace=True)


# ============================================================
# 9. VALIDATION
# ============================================================

print("\nFINAL CHUNK CORPUS")
print("------------------")

print("Total chunks:", len(chunks_df))
print("Sources:", chunks_df["source_id"].nunique())
print(
    "Duplicate IDs:",
    chunks_df["chunk_id"].duplicated().sum()
)
print(
    "Missing text:",
    chunks_df["text"].isnull().sum()
)
print(
    "Empty text:",
    (chunks_df["text"].str.strip() == "").sum()
)
print(
    "Maximum tokens:",
    chunks_df["token_count"].max()
)
print(
    "Chunks exceeding MiniLM limit:",
    (chunks_df["token_count"] > 256).sum()
)

Corpus pages loaded: 1377


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (305 > 256). Running this sequence through the model will result in indexing errors



FINAL CHUNK CORPUS
------------------
Total chunks: 5204
Sources: 5
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Maximum tokens: 220
Chunks exceeding MiniLM limit: 0


In [5]:
# 1. Load corpus
import pandas as pd
import re

corpus_path = "../data/processed/scientific_corpus_pages.csv"
corpus_df = pd.read_csv(corpus_path)

print("Corpus:", corpus_df.shape)

Corpus: (1377, 4)


In [6]:
# 2. Create retrieval dataframe
retrieval_df = corpus_df.copy()

retrieval_df = retrieval_df.dropna(subset=["text"])
retrieval_df["text"] = retrieval_df["text"].astype(str).str.strip()
retrieval_df = retrieval_df[retrieval_df["text"] != ""].copy()
retrieval_df.reset_index(drop=True, inplace=True)

print("Retrieval pages:", len(retrieval_df))

Retrieval pages: 1352


In [7]:
# 3. Clean extracted text
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

retrieval_df["clean_text"] = retrieval_df["text"].apply(clean_text)

print(retrieval_df.columns.tolist())

['source_id', 'filename', 'page_number', 'text', 'clean_text']


In [8]:
import re
import pandas as pd


# ============================================================
# 1. SENTENCE SPLITTING
# ============================================================

def split_into_sentences(text):
    sentences = re.split(
        r'(?<=[.!?])\s+(?=[A-Z0-9])',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# ============================================================
# 2. TOKEN-AWARE CHUNKING
# ============================================================

def create_token_aware_chunks(
    text,
    tokenizer,
    target_tokens=220,
    overlap_sentences=2
):
    sentences = split_into_sentences(text)

    if not sentences:
        return []

    def token_count(value):
        return len(
            tokenizer(
                value,
                add_special_tokens=True,
                truncation=False
            )["input_ids"]
        )

    expanded_sentences = []

    for sentence in sentences:

        if token_count(sentence) <= target_tokens:
            expanded_sentences.append(sentence)
            continue

        token_ids = tokenizer(
            sentence,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        safe_size = target_tokens - 2

        for start in range(0, len(token_ids), safe_size):

            token_piece = token_ids[start:start + safe_size]

            piece = tokenizer.decode(
                token_piece,
                skip_special_tokens=True
            ).strip()

            if piece:
                expanded_sentences.append(piece)

    chunks = []
    current_sentences = []

    for sentence in expanded_sentences:

        candidate = " ".join(
            current_sentences + [sentence]
        ).strip()

        if (
            not current_sentences
            or token_count(candidate) <= target_tokens
        ):
            current_sentences.append(sentence)

        else:
            chunk = " ".join(current_sentences).strip()

            if chunk:
                chunks.append(chunk)

            overlap = (
                current_sentences[-overlap_sentences:]
                if overlap_sentences > 0
                else []
            )

            while overlap:
                candidate = " ".join(
                    overlap + [sentence]
                ).strip()

                if token_count(candidate) <= target_tokens:
                    break

                overlap = overlap[1:]

            current_sentences = overlap + [sentence]

    if current_sentences:
        final_chunk = " ".join(current_sentences).strip()

        if final_chunk:
            chunks.append(final_chunk)

    return chunks


# ============================================================
# 3. CREATE CHUNKS
# ============================================================

all_chunks = []

for _, row in retrieval_df.iterrows():

    page_chunks = create_token_aware_chunks(
        row["clean_text"],
        tokenizer,
        target_tokens=220,
        overlap_sentences=2
    )

    for chunk_index, chunk in enumerate(page_chunks, start=1):

        chunk_id = (
            f'{row["source_id"]}_'
            f'P{int(row["page_number"]):04d}_'
            f'C{chunk_index:03d}'
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "source_id": row["source_id"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": chunk_index,
            "text": chunk
        })


chunks_df = pd.DataFrame(all_chunks)


# ============================================================
# 4. TOKEN COUNTS
# ============================================================

def count_tokens(text):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )


chunks_df["token_count"] = chunks_df["text"].apply(
    count_tokens
)


# ============================================================
# 5. ORIGINAL CREDIT CLEANUP
# ============================================================

credit_mask = (
    chunks_df["text"].str.contains(
        r"^\s*©",
        regex=True,
        na=False
    )
    &
    (chunks_df["token_count"] < 20)
)

before_count = len(chunks_df)

chunks_df = chunks_df[~credit_mask].copy()
chunks_df.reset_index(drop=True, inplace=True)

after_count = len(chunks_df)


# ============================================================
# 6. VALIDATE
# ============================================================

print("Before credit cleanup:", before_count)
print("Credit chunks removed:", before_count - after_count)

print("\nFINAL CHUNK CORPUS")
print("------------------")
print("Total chunks:", len(chunks_df))
print("Sources:", chunks_df["source_id"].nunique())
print("Duplicate IDs:", chunks_df["chunk_id"].duplicated().sum())
print("Missing text:", chunks_df["text"].isnull().sum())
print("Empty text:", (chunks_df["text"].str.strip() == "").sum())
print("Maximum tokens:", chunks_df["token_count"].max())
print(
    "Chunks exceeding MiniLM limit:",
    (chunks_df["token_count"] > 256).sum()
)

Before credit cleanup: 6876
Credit chunks removed: 18

FINAL CHUNK CORPUS
------------------
Total chunks: 6858
Sources: 5
Duplicate IDs: 0
Missing text: 0
Empty text: 0
Maximum tokens: 223
Chunks exceeding MiniLM limit: 0


In [9]:
import re

def classify_noise(text):
    text = str(text).strip()
    lower = text.lower()

    flags = []

    # 1. Very short structural fragments
    if len(text.split()) < 8:
        flags.append("very_short")

    # 2. Explicit reference/bibliography headings
    if re.match(r"^(references|bibliography)\b", lower):
        flags.append("reference_heading")

    # 3. DOI-heavy text — often bibliography/reference material
    doi_count = len(
        re.findall(
            r"\bdoi\b|https?://doi\.org",
            lower
        )
    )

    if doi_count >= 2:
        flags.append("doi_heavy")

    # 4. Questionnaire / survey-like material
    survey_terms = [
        "questionnaire",
        "respondent",
        "please indicate",
        "please specify",
        "how do you perceive",
        "national survey"
    ]

    survey_matches = sum(
        term in lower
        for term in survey_terms
    )

    if survey_matches >= 2:
        flags.append("questionnaire")

    return flags


chunks_df["noise_flags"] = chunks_df["text"].apply(
    classify_noise
)

noise_candidates = chunks_df[
    chunks_df["noise_flags"].apply(len) > 0
].copy()

print("Total chunks:", len(chunks_df))
print("Potential noise chunks:", len(noise_candidates))

print("\nNoise type counts:")

all_flags = [
    flag
    for flags in noise_candidates["noise_flags"]
    for flag in flags
]

print(pd.Series(all_flags).value_counts())

display(
    noise_candidates[
        [
            "chunk_id",
            "source_id",
            "page_number",
            "token_count",
            "noise_flags",
            "text"
        ]
    ].head(30)
)

Total chunks: 6858
Potential noise chunks: 183

Noise type counts:
doi_heavy            96
reference_heading    76
very_short           15
questionnaire         4
Name: count, dtype: int64


,chunk_id,source_id,page_number,token_count,noise_flags,text
24,SRC01_P0012_C002,SRC01,12,11,[very_short],responded to the survey 445 references 447
66,SRC01_P0028_C004,SRC01,28,14,[very_short],##cados pncti | technology and innovation plan
1065,SRC01_P0280_C003,SRC01,280,5,[very_short],low low low
1510,SRC01_P0393_C006,SRC01,393,11,[very_short],##tkatalog _ node. html
1805,SRC01_P0465_C001,SRC01,465,3,[very_short],435
1810,SRC01_P0469_C001,SRC01,469,201,[questionnaire],National Survey on Status of Soil Biodiversity...
1814,SRC01_P0471_C001,SRC01,471,200,[questionnaire],National Survey on Status of Soil Biodiversity...
1819,SRC01_P0473_C001,SRC01,473,218,[questionnaire],National Survey on Status of Soil Biodiversity...
1821,SRC01_P0477_C001,SRC01,477,211,[reference_heading],"References 447 REFERENCES A’Bear A.D., Johnson..."
1823,SRC01_P0477_C003,SRC01,477,211,[doi_heavy],"Abinandan, S., Subashchandrabose, S.R., Venkat..."


In [10]:
# ============================================================
# CREATE CLEAN RETRIEVAL CORPUS
# ============================================================

SAFE_NOISE_FLAGS = {
    "very_short",
    "reference_heading",
    "questionnaire"
}

def should_remove(flags):
    return any(
        flag in SAFE_NOISE_FLAGS
        for flag in flags
    )


remove_mask = chunks_df["noise_flags"].apply(
    should_remove
)

removed_chunks_df = chunks_df[
    remove_mask
].copy()

retrieval_chunks_df = chunks_df[
    ~remove_mask
].copy()

retrieval_chunks_df.reset_index(
    drop=True,
    inplace=True
)


print("Original chunks:", len(chunks_df))
print("Removed chunks:", len(removed_chunks_df))
print("Retrieval chunks:", len(retrieval_chunks_df))

print("\nRemoved by reason:")

removed_flags = [
    flag
    for flags in removed_chunks_df["noise_flags"]
    for flag in flags
    if flag in SAFE_NOISE_FLAGS
]

print(pd.Series(removed_flags).value_counts())

Original chunks: 6858
Removed chunks: 95
Retrieval chunks: 6763

Removed by reason:
reference_heading    76
very_short           15
questionnaire         4
Name: count, dtype: int64


In [16]:
def decompose_query(question):
    q = question.lower()

    queries = {
        "original": question
    }

    # ==================================================
    # SOC / SOIL HEALTH
    # ==================================================

    if any(term in q for term in [
        "soil carbon",
        "soil organic carbon",
        "soc"
    ]):

        if any(term in q for term in [
            "very low",
            "low soil carbon",
            "low soil organic carbon",
            "soc 0.3",
            "0.3%",
            "0.4%"
        ]):
            queries["soc"] = (
                f"{question} "
                "Effects of low soil organic carbon on soil health, "
                "soil biodiversity, soil organisms and ecosystem functioning."
            )

        elif any(term in q for term in [
            "good soil carbon",
            "high soil carbon"
        ]):
            queries["soc"] = (
                f"{question} "
                "Healthy soil organic carbon, soil biodiversity, "
                "soil ecosystem stability and maintenance."
            )

        else:
            queries["soc"] = (
                f"{question} "
                "Relationship between soil organic carbon, "
                "soil biodiversity and soil health."
            )

    # ==================================================
    # RAINFALL / WATER
    # ==================================================

    if any(term in q for term in [
        "rainfall",
        "semi-arid",
        "semi arid",
        "dryland",
        "drought"
    ]):

        if any(term in q for term in [
            "low rainfall",
            "semi-arid",
            "semi arid",
            "dryland",
            "drought"
        ]):
            queries["rainfall"] = (
                f"{question} "
                "Low rainfall semi-arid dryland restoration, "
                "water limitation, soil water availability "
                "and vegetation establishment."
            )

        elif (
    "high rainfall" in q
    or "rainfall is high" in q
    or "rainfall here is high" in q
        ):
            queries["rainfall"] = (
                f"{question} "
                "High rainfall non-water-limited agricultural "
                "soil restoration, vegetation and management practices."
          )

        elif any(term in q for term in [
            "decent rainfall",
            "adequate rainfall"
        ]):
            queries["rainfall"] = (
                f"{question} "
                "Adequate rainfall and water availability for "
                "agricultural soil restoration and vegetation establishment."
            )

        else:
            queries["rainfall"] = (
                f"{question} "
                "Rainfall, water availability and restoration."
            )

    # ==================================================
    # LAND USE
    # ==================================================

    if any(term in q for term in [
        "monoculture",
        "wheat",
        "corn",
        "maize",
        "agroforestry"
    ]):

        if "agroforestry" in q:
            queries["land_use"] = (
                f"{question} "
                "Existing agroforestry systems, biodiversity, "
                "soil health, ecosystem services and "
                "sustainable maintenance."
            )

        elif "wheat" in q:
            queries["land_use"] = (
                f"{question} "
                "Wheat monoculture, crop diversification, "
                "crop rotation, intercropping, agroforestry "
                "and biodiversity."
            )

        elif "corn" in q or "maize" in q:
            queries["land_use"] = (
                f"{question} "
                "Corn maize monoculture, crop diversification, "
                "cover crops, intercropping and soil biodiversity."
            )

        else:
            queries["land_use"] = (
                f"{question} "
                "Monoculture, agricultural diversification "
                "and biodiversity."
            )

    # ==================================================
    # BIODIVERSITY
    # ==================================================

    if any(term in q for term in [
        "biodiversity",
        "species",
        "species count",
        "species diversity"
    ]):

        if any(term in q for term in [
            "declining",
            "decline",
            "low species",
            "low species count"
        ]):
            queries["biodiversity"] = (
                f"{question} "
                "Declining or low biodiversity, causes of "
                "biodiversity loss and agricultural practices "
                "that restore biodiversity."
            )

        elif any(term in q for term in [
            "high species",
            "species counts are high",
            "diversity seems fine",
            "moderate",
            "stable"
        ]):
            queries["biodiversity"] = (
                f"{question} "
                "Maintaining existing agricultural biodiversity, "
                "ecosystem stability and sustainable management."
            )

        else:
            queries["biodiversity"] = (
                f"{question} "
                "Agricultural biodiversity, ecosystem health "
                "and biodiversity-friendly management."
            )

    return queries

In [14]:
for case_id in ["GC01", "GC02", "GC04", "GC06", "GC07"]:

    question = scientific_eval_df.loc[
        scientific_eval_df["case_id"] == case_id,
        "question"
    ].iloc[0]

    print("\n" + "=" * 80)
    print(case_id)
    print("=" * 80)

    queries = decompose_query(question)

    for aspect, query in queries.items():
        print(f"\n{aspect.upper()}:")
        print(query)

NameError: name 'scientific_eval_df' is not defined

In [17]:
test_cases = {
    "GC01": (
        "A farm has low soil organic carbon, low rainfall, "
        "monoculture wheat, and low species counts. "
        "What intervention could improve its condition?"
    ),

    "GC02": (
        "My land already has good soil carbon, decent rainfall, "
        "and agroforestry. Species counts are high. "
        "Is there anything more I should do?"
    ),

    "GC04": (
        "We get decent rainfall but our soil carbon is low "
        "and we grow only wheat. What should we do?"
    ),

    "GC06": (
        "Rainfall here is high, but soil carbon is very low "
        "and it's monoculture corn. Species diversity seems fine for now."
    ),

    "GC07": (
        "Semi-arid region, soil organic carbon 0.3%, rainfall is low, "
        "crop is monoculture wheat. "
        "What should I do to improve biodiversity and soil health?"
    )
}


for case_id, question in test_cases.items():

    print("\n" + "=" * 80)
    print(case_id)
    print("=" * 80)

    queries = decompose_query(question)

    for aspect, query in queries.items():
        print(f"\n{aspect.upper()}:")
        print(query)


GC01

ORIGINAL:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

SOC:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Effects of low soil organic carbon on soil health, soil biodiversity, soil organisms and ecosystem functioning.

RAINFALL:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Low rainfall semi-arid dryland restoration, water limitation, soil water availability and vegetation establishment.

LAND_USE:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Wheat monoculture, crop diversification, crop rotation, intercropping, agroforestry and biodiversity.

BIODIVERSITY:
A farm has low soil organic carbon, low rainfall, 

In [18]:
texts = retrieval_chunks_df["text"].tolist()

print("Chunks to embed:", len(texts))

clean_embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding generation complete.")
print("Embedding matrix shape:", clean_embeddings.shape)

Chunks to embed: 6763


Batches:   0%|          | 0/212 [00:00<?, ?it/s]


Embedding generation complete.
Embedding matrix shape: (6763, 384)


In [19]:
import numpy as np

print("Number of chunks:", len(retrieval_chunks_df))
print("Number of embeddings:", len(clean_embeddings))
print("Embedding dimensions:", clean_embeddings.shape[1])
print("Contains NaN:", np.isnan(clean_embeddings).any())
print("Contains infinity:", np.isinf(clean_embeddings).any())

Number of chunks: 6763
Number of embeddings: 6763
Embedding dimensions: 384
Contains NaN: False
Contains infinity: False


In [20]:
import chromadb
import os

# Persistent ChromaDB folder
chroma_path = "../chroma_db"

client = chromadb.PersistentClient(
    path=chroma_path
)

collection_name = "biodiversity_knowledge_clean"

# Delete ONLY this collection if it already exists
try:
    client.delete_collection(collection_name)
    print("Old clean collection deleted.")
except Exception:
    pass

# Create fresh collection
collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

print("Created collection:", collection_name)

Created collection: biodiversity_knowledge_clean


In [21]:
batch_size = 500

for start in range(0, len(retrieval_chunks_df), batch_size):

    end = min(
        start + batch_size,
        len(retrieval_chunks_df)
    )

    batch_df = retrieval_chunks_df.iloc[start:end]

    batch_embeddings = clean_embeddings[start:end]

    collection.add(
        ids=batch_df["chunk_id"].tolist(),

        documents=batch_df["text"].tolist(),

        embeddings=batch_embeddings.tolist(),

        metadatas=[
            {
                "source_id": row["source_id"],
                "filename": row["filename"],
                "page_number": int(row["page_number"]),
                "chunk_index": int(row["chunk_index"])
            }
            for _, row in batch_df.iterrows()
        ]
    )

    print(
        f"Stored {end}/{len(retrieval_chunks_df)} chunks"
    )

Stored 500/6763 chunks
Stored 1000/6763 chunks
Stored 1500/6763 chunks
Stored 2000/6763 chunks
Stored 2500/6763 chunks
Stored 3000/6763 chunks
Stored 3500/6763 chunks
Stored 4000/6763 chunks
Stored 4500/6763 chunks
Stored 5000/6763 chunks
Stored 5500/6763 chunks
Stored 6000/6763 chunks
Stored 6500/6763 chunks
Stored 6763/6763 chunks


In [22]:
print("Expected chunks:", len(retrieval_chunks_df))
print("Stored in Chroma:", collection.count())

assert collection.count() == len(retrieval_chunks_df)

print("\nClean ChromaDB validation passed.")

Expected chunks: 6763
Stored in Chroma: 6763

Clean ChromaDB validation passed.


In [23]:
def retrieve_all_aspects(question, per_aspect_k=3):
    """
    Search every relevant aspect of the question.

    Returns candidates from ALL aspects instead of stopping
    after the first top_k results.
    """

    decomposed = decompose_query(question)

    candidates = {}

    for aspect, query in decomposed.items():

        # Embed this aspect-specific query
        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        # Search clean Chroma collection
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=per_aspect_k,
            include=[
                "documents",
                "metadatas",
                "distances"
            ]
        )

        ids = results["ids"][0]
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        for rank, (chunk_id, document, metadata, distance) in enumerate(
            zip(ids, documents, metadatas, distances),
            start=1
        ):

            # Deduplicate same chunk appearing for multiple aspects
            if chunk_id not in candidates:
                candidates[chunk_id] = {
                    "chunk_id": chunk_id,
                    "text": document,
                    "source_id": metadata["source_id"],
                    "page_number": metadata["page_number"],
                    "distance": float(distance),
                    "best_distance": float(distance),
                    "matched_aspects": [aspect],
                    "aspect_ranks": {aspect: rank}
                }

            else:
                candidate = candidates[chunk_id]

                candidate["matched_aspects"].append(aspect)
                candidate["aspect_ranks"][aspect] = rank

                candidate["best_distance"] = min(
                    candidate["best_distance"],
                    float(distance)
                )

    return list(candidates.values())

In [24]:
question = test_cases["GC01"]

candidates = retrieve_all_aspects(
    question,
    per_aspect_k=3
)

print("Unique candidates:", len(candidates))

for aspect in decompose_query(question):

    print("\n" + "=" * 80)
    print("ASPECT:", aspect.upper())
    print("=" * 80)

    aspect_results = [
        item
        for item in candidates
        if aspect in item["matched_aspects"]
    ]

    aspect_results.sort(
        key=lambda x: x["aspect_ranks"][aspect]
    )

    for item in aspect_results:
        print(
            item["aspect_ranks"][aspect],
            item["chunk_id"],
            item["source_id"],
            "Page", item["page_number"],
            "Distance", round(item["best_distance"], 4)
        )

        print(item["text"][:500])
        print()

Unique candidates: 11

ASPECT: ORIGINAL
1 SRC01_P0270_C005 SRC01 Page 270 Distance 0.3153
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low 

2 SRC02_P0272_C003 SRC02 Page 272 Distance 0.3883
The consequences of this have included decreases in soil organic matter and high discharge of nutrients into the environment in areas where large numbers of animals are raised in intensive units, with neg- ative impacts in turn on aquatic, soil and other biodiversity (see also Chapter 3). Time-series data for such changes are rare. However, data from the a

In [25]:
def retrieve_balanced_top3(question):
    candidates = retrieve_all_aspects(
        question,
        per_aspect_k=3
    )

    decomposed = decompose_query(question)

    # Original is useful for discovery, but the scientific
    # aspects are more useful for coverage selection.
    aspects = [
        aspect
        for aspect in [
            "soc",
            "rainfall",
            "land_use",
            "biodiversity"
        ]
        if aspect in decomposed
    ]

    selected = []
    used_ids = set()

    # --------------------------------------------------
    # PASS 1:
    # Best unique chunk from each scientific aspect
    # --------------------------------------------------

    for aspect in aspects:

        aspect_candidates = [
            item
            for item in candidates
            if (
                aspect in item["matched_aspects"]
                and item["chunk_id"] not in used_ids
            )
        ]

        if not aspect_candidates:
            continue

        aspect_candidates.sort(
            key=lambda x: x["aspect_ranks"][aspect]
        )

        best = aspect_candidates[0]

        selected.append({
            **best,
            "selected_for": aspect
        })

        used_ids.add(best["chunk_id"])

    # --------------------------------------------------
    # If >3 aspects exist, rank selected evidence
    # by best semantic distance.
    # --------------------------------------------------

    if len(selected) > 3:
        selected.sort(
            key=lambda x: x["best_distance"]
        )
        selected = selected[:3]

    # --------------------------------------------------
    # PASS 2:
    # If fewer than 3, fill remaining positions using
    # best unused candidates from all searches.
    # --------------------------------------------------

    if len(selected) < 3:

        remaining = [
            item
            for item in candidates
            if item["chunk_id"] not in used_ids
        ]

        remaining.sort(
            key=lambda x: x["best_distance"]
        )

        for item in remaining:

            selected.append({
                **item,
                "selected_for": "fill"
            })

            used_ids.add(item["chunk_id"])

            if len(selected) == 3:
                break

    return selected

In [26]:
results = retrieve_balanced_top3(
    test_cases["GC01"]
)

for rank, item in enumerate(results, start=1):

    print("\n" + "=" * 90)
    print("RESULT", rank)
    print("=" * 90)

    print("Selected for:", item["selected_for"])
    print("Chunk:", item["chunk_id"])
    print("Source:", item["source_id"])
    print("Page:", item["page_number"])
    print("Distance:", round(item["best_distance"], 4))
    print("Matched aspects:", item["matched_aspects"])

    print("\nTEXT:")
    print(item["text"])


RESULT 1
Selected for: rainfall
Chunk: SRC01_P0270_C005
Source: SRC01
Page: 270
Distance: 0.3153
Matched aspects: ['original', 'rainfall', 'land_use']

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low areas (Table 4.3.1.1).

RESULT 2
Selected for: biodiversity
Chunk: SRC02_P0241_C001
Source: SRC02
Page: 241
Distance: 0.3328
Matched aspects: ['biodiversity']

TEXT:
197 THE STATE OF USE OF BIODIVERSITY FOR FOOD AND AGRICULTURE 5 the state OF THE WORLD'S biodiversity FOr FOOD AND AGRICULTURE Figure 5.1 Perceived impacts on biodiversity for

In [27]:
import re
import pandas as pd


def looks_like_bibliography(text):
    text = str(text).strip()

    # Common bibliography characteristics
    year_matches = re.findall(
        r"\b(?:19|20)\d{2}[a-z]?\b",
        text
    )

    doi_matches = re.findall(
        r"(?:doi\s*:|https?://doi\.org/)",
        text,
        flags=re.IGNORECASE
    )

    # Author-list patterns such as:
    # Smith, J. & Jones, A.
    author_patterns = re.findall(
        r"\b[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÿ'’-]+,\s*"
        r"[A-Z](?:\.[A-Z])?\.",
        text
    )

    # Journal/reference-style vocabulary
    reference_terms = re.findall(
        r"\b(?:journal|vol\.?|volume|pp\.?|"
        r"proceedings|springer|elsevier|"
        r"wiley|cambridge university press)\b",
        text,
        flags=re.IGNORECASE
    )

    score = 0

    if len(year_matches) >= 3:
        score += 1

    if len(author_patterns) >= 3:
        score += 1

    if len(doi_matches) >= 2:
        score += 1

    if len(reference_terms) >= 2:
        score += 1

    # Require multiple bibliography signals.
    return score >= 2


retrieval_chunks_df["bibliography_like"] = (
    retrieval_chunks_df["text"].apply(looks_like_bibliography)
)

bib_candidates = retrieval_chunks_df[
    retrieval_chunks_df["bibliography_like"]
].copy()


print("Current retrieval chunks:", len(retrieval_chunks_df))
print("Bibliography-like candidates:", len(bib_candidates))

display(
    bib_candidates[
        [
            "chunk_id",
            "source_id",
            "page_number",
            "token_count",
            "text"
        ]
    ].head(30)
)

Current retrieval chunks: 6763
Bibliography-like candidates: 957


,chunk_id,source_id,page_number,token_count,text
678,SRC01_P0192_C003,SRC01,192,213,3.5.6 | WASTEWATER TREATMENT Ecosystems such a...
679,SRC01_P0192_C004,SRC01,192,166,A “solids” component comprised of biological c...
1813,SRC01_P0477_C002,SRC01,477,199,Commercialisation of entomopathogenic nematode...
1814,SRC01_P0477_C003,SRC01,477,211,"Abinandan, S., Subashchandrabose, S.R., Venkat..."
1815,SRC01_P0477_C004,SRC01,477,217,Effect of industrial residues (raw and compost...
1816,SRC01_P0477_C005,SRC01,477,139,2011. Emergence or self-organization?: Look to...
1818,SRC01_P0478_C002,SRC01,478,215,2018. Revisiting risk governance of GM plants:...
1819,SRC01_P0478_C003,SRC01,478,219,2011. Role of arbuscular mycorrhizal fungi (AM...
1820,SRC01_P0478_C004,SRC01,478,213,Agronet.fi [online]. [Cited 2 October 2020]. h...
1821,SRC01_P0478_C005,SRC01,478,207,Balance Social 2018. [online]. [Cited 5 Octobe...


In [28]:
bib_candidates.groupby("source_id")["page_number"].agg(
    ["min", "max", "count"]
)

,min,max,count
source_id,,,
SRC01,192,612,512
SRC02,497,573,413
SRC03,149,168,23
SRC05,8,8,9


In [29]:
for source_id in sorted(
    bib_candidates["source_id"].unique()
):

    pages = sorted(
        bib_candidates.loc[
            bib_candidates["source_id"] == source_id,
            "page_number"
        ].unique()
    )

    print("\n", source_id)
    print(pages)


 SRC01
[np.int64(192), np.int64(477), np.int64(478), np.int64(479), np.int64(480), np.int64(481), np.int64(482), np.int64(483), np.int64(484), np.int64(485), np.int64(486), np.int64(487), np.int64(488), np.int64(489), np.int64(490), np.int64(491), np.int64(492), np.int64(493), np.int64(494), np.int64(495), np.int64(496), np.int64(497), np.int64(498), np.int64(499), np.int64(500), np.int64(501), np.int64(502), np.int64(503), np.int64(504), np.int64(505), np.int64(506), np.int64(507), np.int64(508), np.int64(509), np.int64(510), np.int64(511), np.int64(512), np.int64(513), np.int64(514), np.int64(515), np.int64(516), np.int64(517), np.int64(518), np.int64(519), np.int64(520), np.int64(521), np.int64(522), np.int64(523), np.int64(524), np.int64(525), np.int64(526), np.int64(528), np.int64(529), np.int64(530), np.int64(531), np.int64(532), np.int64(533), np.int64(534), np.int64(535), np.int64(536), np.int64(537), np.int64(538), np.int64(539), np.int64(540), np.int64(541), np.int64(542), n

In [30]:
# ============================================================
# REMOVE CONFIRMED BIBLIOGRAPHY SECTIONS
# ============================================================

before_bib_cleanup = len(retrieval_chunks_df)

reference_section_mask = (
    # SRC01: references begin at page 477
    (
        (retrieval_chunks_df["source_id"] == "SRC01")
        & (retrieval_chunks_df["page_number"] >= 477)
    )
    |
    # SRC02: references begin around page 497
    (
        (retrieval_chunks_df["source_id"] == "SRC02")
        & (retrieval_chunks_df["page_number"] >= 497)
    )
)

removed_reference_chunks = retrieval_chunks_df[
    reference_section_mask
].copy()

retrieval_chunks_df = retrieval_chunks_df[
    ~reference_section_mask
].copy()

retrieval_chunks_df.reset_index(drop=True, inplace=True)

print("Before bibliography cleanup:", before_bib_cleanup)
print("Reference chunks removed:", len(removed_reference_chunks))
print("Final retrieval chunks:", len(retrieval_chunks_df))

print("\nRemoved by source:")
print(
    removed_reference_chunks["source_id"]
    .value_counts()
)

Before bibliography cleanup: 6763
Reference chunks removed: 1316
Final retrieval chunks: 5447

Removed by source:
source_id
SRC01    730
SRC02    586
Name: count, dtype: int64


In [32]:
print("\nMaximum remaining pages:")
print(
    retrieval_chunks_df.groupby("source_id")["page_number"].max()
)

print("\nRemaining suspicious bibliography candidates:")

remaining_bib = bib_candidates[
    bib_candidates["chunk_id"].isin(
        retrieval_chunks_df["chunk_id"]
    )
]

print(
    remaining_bib.groupby("source_id")["page_number"]
    .agg(["min", "max", "count"])
)

display(
    remaining_bib[
        ["chunk_id", "source_id", "page_number", "text"]
    ].head(30)
)


Maximum remaining pages:
source_id
SRC01    475
SRC02    496
SRC03    172
SRC04      5
SRC05      8
Name: page_number, dtype: int64

Remaining suspicious bibliography candidates:
           min  max  count
source_id                 
SRC01      192  192      2
SRC03      149  168     23
SRC05        8    8      9


,chunk_id,source_id,page_number,text
678,SRC01_P0192_C003,SRC01,192,3.5.6 | WASTEWATER TREATMENT Ecosystems such a...
679,SRC01_P0192_C004,SRC01,192,A “solids” component comprised of biological c...
3057,SRC03_P0149_C001,SRC03,149,"127 References, further reading, tools and gui..."
3058,SRC03_P0149_C002,SRC03,149,"Aronson, (eds). Restoration ecology: the new f..."
3060,SRC03_P0150_C001,SRC03,150,Global guidelines for the restoration of degra...
3061,SRC03_P0150_C002,SRC03,150,"Buffle, P. & Reij, C. 2012. Land rehabilitatio..."
3063,SRC03_P0150_C004,SRC03,150,Comité permanent Inter-États de lutte contre l...
3064,SRC03_P0151_C002,SRC03,151,Making change happen: what can governments do ...
3073,SRC03_P0154_C001,SRC03,154,Global guidelines for the restoration of degra...
3076,SRC03_P0154_C004,SRC03,154,2011. Dry land tree management for improved ho...


In [33]:
# ============================================================
# FINAL STRUCTURAL BACK-MATTER CLEANUP
# ============================================================

before_final_cleanup = len(retrieval_chunks_df)

final_backmatter_mask = (
    # SRC03: references/further reading/back matter starts page 149
    (
        (retrieval_chunks_df["source_id"] == "SRC03")
        & (retrieval_chunks_df["page_number"] >= 149)
    )
    |
    # SRC05: page 8 is the paper's reference section
    (
        (retrieval_chunks_df["source_id"] == "SRC05")
        & (retrieval_chunks_df["page_number"] >= 8)
    )
)

removed_final_backmatter = retrieval_chunks_df[
    final_backmatter_mask
].copy()

retrieval_chunks_df = retrieval_chunks_df[
    ~final_backmatter_mask
].copy()

retrieval_chunks_df.reset_index(drop=True, inplace=True)

print("Before final cleanup:", before_final_cleanup)
print("Additional chunks removed:", len(removed_final_backmatter))
print("FINAL retrieval corpus:", len(retrieval_chunks_df))

print("\nRemoved by source:")
print(removed_final_backmatter["source_id"].value_counts())

print("\nMaximum remaining page per source:")
print(
    retrieval_chunks_df
    .groupby("source_id")["page_number"]
    .max()
)

Before final cleanup: 5447
Additional chunks removed: 83
FINAL retrieval corpus: 5364

Removed by source:
source_id
SRC03    66
SRC05    17
Name: count, dtype: int64

Maximum remaining page per source:
source_id
SRC01    475
SRC02    496
SRC03    147
SRC04      5
SRC05      7
Name: page_number, dtype: int64


In [34]:
print("FINAL CORPUS VALIDATION")
print("-----------------------")
print("Total chunks:", len(retrieval_chunks_df))
print("Sources:", retrieval_chunks_df["source_id"].nunique())
print("Duplicate IDs:", retrieval_chunks_df["chunk_id"].duplicated().sum())
print("Missing text:", retrieval_chunks_df["text"].isna().sum())
print("Empty text:", (retrieval_chunks_df["text"].str.strip() == "").sum())

# Confirm known bad chunks are gone
known_bad = [
    "SRC01_P0601_C005",
    "SRC02_P0501_C008",
    "SRC03_P0149_C001",
    "SRC05_P0008_C001"
]

print("\nKnown bad chunk check:")
for chunk_id in known_bad:
    print(
        chunk_id,
        "PRESENT" if chunk_id in set(retrieval_chunks_df["chunk_id"])
        else "REMOVED ✓"
    )

FINAL CORPUS VALIDATION
-----------------------
Total chunks: 5364
Sources: 5
Duplicate IDs: 0
Missing text: 0
Empty text: 0

Known bad chunk check:
SRC01_P0601_C005 REMOVED ✓
SRC02_P0501_C008 REMOVED ✓
SRC03_P0149_C001 REMOVED ✓
SRC05_P0008_C001 REMOVED ✓


In [35]:
final_texts = retrieval_chunks_df["text"].tolist()

print("Chunks to embed:", len(final_texts))

final_embeddings = embedding_model.encode(
    final_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding generation complete.")
print("Embedding matrix shape:", final_embeddings.shape)

Chunks to embed: 5364


Batches:   0%|          | 0/168 [00:00<?, ?it/s]


Embedding generation complete.
Embedding matrix shape: (5364, 384)


In [36]:
import numpy as np

print("Chunks:", len(retrieval_chunks_df))
print("Embeddings:", len(final_embeddings))
print("Dimensions:", final_embeddings.shape[1])
print("Contains NaN:", np.isnan(final_embeddings).any())
print("Contains infinity:", np.isinf(final_embeddings).any())

Chunks: 5364
Embeddings: 5364
Dimensions: 384
Contains NaN: False
Contains infinity: False


In [37]:
collection_name = "biodiversity_knowledge_clean"

try:
    client.delete_collection(collection_name)
    print("Old collection deleted.")
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

print("Fresh collection created.")

Old collection deleted.
Fresh collection created.


In [38]:
batch_size = 500

for start in range(0, len(retrieval_chunks_df), batch_size):

    end = min(
        start + batch_size,
        len(retrieval_chunks_df)
    )

    batch_df = retrieval_chunks_df.iloc[start:end]
    batch_embeddings = final_embeddings[start:end]

    collection.add(
        ids=batch_df["chunk_id"].tolist(),
        documents=batch_df["text"].tolist(),
        embeddings=batch_embeddings.tolist(),
        metadatas=[
            {
                "source_id": row["source_id"],
                "filename": row["filename"],
                "page_number": int(row["page_number"]),
                "chunk_index": int(row["chunk_index"])
            }
            for _, row in batch_df.iterrows()
        ]
    )

    print(f"Stored {end}/{len(retrieval_chunks_df)} chunks")

Stored 500/5364 chunks
Stored 1000/5364 chunks
Stored 1500/5364 chunks
Stored 2000/5364 chunks
Stored 2500/5364 chunks
Stored 3000/5364 chunks
Stored 3500/5364 chunks
Stored 4000/5364 chunks
Stored 4500/5364 chunks
Stored 5000/5364 chunks
Stored 5364/5364 chunks


In [39]:
print("Expected:", len(retrieval_chunks_df))
print("Stored:", collection.count())

assert collection.count() == len(retrieval_chunks_df)

print("\nFINAL ChromaDB validation passed.")

Expected: 5364
Stored: 5364

FINAL ChromaDB validation passed.


In [40]:
results = retrieve_balanced_top3(
    test_cases["GC01"]
)

for rank, item in enumerate(results, start=1):

    print("\n" + "=" * 90)
    print("RESULT", rank)
    print("=" * 90)

    print("Selected for:", item["selected_for"])
    print("Chunk:", item["chunk_id"])
    print("Source:", item["source_id"])
    print("Page:", item["page_number"])
    print("Distance:", round(item["best_distance"], 4))
    print("Matched aspects:", item["matched_aspects"])

    print("\nTEXT:")
    print(item["text"])


RESULT 1
Selected for: rainfall
Chunk: SRC01_P0270_C005
Source: SRC01
Page: 270
Distance: 0.3153
Matched aspects: ['original', 'rainfall', 'land_use']

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low areas (Table 4.3.1.1).

RESULT 2
Selected for: biodiversity
Chunk: SRC02_P0241_C001
Source: SRC02
Page: 241
Distance: 0.3328
Matched aspects: ['biodiversity']

TEXT:
197 THE STATE OF USE OF BIODIVERSITY FOR FOOD AND AGRICULTURE 5 the state OF THE WORLD'S biodiversity FOr FOOD AND AGRICULTURE Figure 5.1 Perceived impacts on biodiversity for

In [41]:
all_golden_results = {}

for case_id, question in test_cases.items():

    print("\n" + "#" * 100)
    print(case_id)
    print("QUESTION:", question)
    print("#" * 100)

    results = retrieve_balanced_top3(question)

    all_golden_results[case_id] = results

    for rank, item in enumerate(results, start=1):

        print("\n" + "=" * 80)
        print(f"RESULT {rank}")
        print("=" * 80)

        print("Selected for:", item["selected_for"])
        print("Chunk:", item["chunk_id"])
        print("Source:", item["source_id"])
        print("Page:", item["page_number"])
        print("Distance:", round(item["best_distance"], 4))
        print("Matched aspects:", item["matched_aspects"])

        print("\nTEXT:")
        print(item["text"])


####################################################################################################
GC01
QUESTION: A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?
####################################################################################################

RESULT 1
Selected for: rainfall
Chunk: SRC01_P0270_C005
Source: SRC01
Page: 270
Distance: 0.3153
Matched aspects: ['original', 'rainfall', 'land_use']

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and wat

In [42]:
question = test_cases["GC07"]

queries = decompose_query(question)

for aspect, query in queries.items():

    print("\n" + "=" * 90)
    print("ASPECT:", aspect.upper())
    print("=" * 90)

    results = retrieve(query, top_k=10)

    for rank, item in enumerate(results, start=1):

        print(
            rank,
            item["chunk_id"],
            item["source_id"],
            "Page", item["page_number"],
            "Distance", round(item["distance"], 4)
        )

        print(item["text"][:350].replace("\n", " "))
        print()


ASPECT: ORIGINAL


NameError: name 'retrieve' is not defined

In [44]:
# ============================================================
# GC07 DIAGNOSTIC — TOP 10 PER ASPECT
# ============================================================

question = test_cases["GC07"]

queries = decompose_query(question)

for aspect, query in queries.items():

    print("\n" + "=" * 90)
    print("ASPECT:", aspect.upper())
    print("=" * 90)

    # Embed query using the SAME MiniLM model
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Search final Chroma collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10,
        include=["documents", "metadatas", "distances"]
    )

    for rank in range(len(results["ids"][0])):

        chunk_id = results["ids"][0][rank]
        text = results["documents"][0][rank]
        metadata = results["metadatas"][0][rank]
        distance = results["distances"][0][rank]

        print(
            f"\n{rank + 1}. "
            f"{chunk_id} | "
            f"{metadata['source_id']} | "
            f"Page {metadata['page_number']} | "
            f"Distance {distance:.4f}"
        )

        print(text[:500])


ASPECT: ORIGINAL

1. SRC01_P0344_C001 | SRC01 | Page 344 | Distance 0.2901
State of knowledge of soil biodiversity 314 10 0 20 What are the major practices in this country that might have negatively impacted soil biodiversity in the last 10 years (1 being the lowest important and 5 being the highest)?

2. SRC01_P0270_C005 | SRC01 | Page 270 | Distance 0.3163
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low 

3. SRC01_P0449_C004 | SRC01 | Page 449 | Distance 0.3310
For example, the commercial inoculation of crop legumes grown with selected eff

In [45]:
import re


# ============================================================
# EVIDENCE QUALITY PENALTY
# ============================================================

def evidence_quality_penalty(text):
    """
    Lower = better.

    Adds penalties to chunks that are semantically similar
    but structurally poor evidence, such as survey questions,
    TOC fragments, figure-heavy chunks, etc.
    """

    t = text.lower().strip()

    penalty = 0.0

    # Survey / questionnaire material
    survey_patterns = [
        "survey question",
        "countries that replied",
        "please select",
        "please provide additional information",
        "what are the major practices in this country",
        "country responses to the soil biodiversity survey"
    ]

    if any(pattern in t for pattern in survey_patterns):
        penalty += 0.12

    # Table of contents / navigation fragments
    toc_patterns = [
        "table of contents",
        "chapter 1",
        "chapter 2",
        "chapter 3",
        "chapter 4",
        "chapter 5",
        "chapter 6",
        "chapter 7",
        "annex i"
    ]

    if any(pattern in t for pattern in toc_patterns):
        penalty += 0.10

    # Figure/table extraction rather than explanatory prose
    if (
        ("figure " in t or "table " in t)
        and len(t.split()) < 180
    ):
        penalty += 0.035

    # Question-heavy text
    question_count = text.count("?")

    if question_count >= 2:
        penalty += 0.08
    elif question_count == 1:
        penalty += 0.025

    return penalty

In [46]:
def retrieve_candidates(question, per_aspect=10):

    queries = decompose_query(question)

    candidates = {}

    for aspect, query in queries.items():

        query_embedding = embedding_model.encode(
            query,
            normalize_embeddings=True
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=per_aspect,
            include=["documents", "metadatas", "distances"]
        )

        for i, chunk_id in enumerate(results["ids"][0]):

            text = results["documents"][0][i]
            metadata = results["metadatas"][0][i]
            distance = results["distances"][0][i]

            if chunk_id not in candidates:

                candidates[chunk_id] = {
                    "chunk_id": chunk_id,
                    "text": text,
                    "source_id": metadata["source_id"],
                    "page_number": metadata["page_number"],
                    "distances": {},
                    "matched_aspects": []
                }

            candidates[chunk_id]["distances"][aspect] = distance
            candidates[chunk_id]["matched_aspects"].append(aspect)

    return candidates

In [47]:
def rerank_candidates(question, per_aspect=10):

    candidates = retrieve_candidates(
        question,
        per_aspect=per_aspect
    )

    reranked = []

    for item in candidates.values():

        best_distance = min(item["distances"].values())

        penalty = evidence_quality_penalty(
            item["text"]
        )

        final_score = best_distance + penalty

        item["best_distance"] = best_distance
        item["quality_penalty"] = penalty
        item["final_score"] = final_score

        reranked.append(item)

    reranked.sort(
        key=lambda x: x["final_score"]
    )

    return reranked

In [48]:
def retrieve_quality_balanced_top3(question):

    candidates = rerank_candidates(
        question,
        per_aspect=10
    )

    selected = []
    selected_ids = set()

    # --------------------------------------------------
    # 1. Rainfall
    # --------------------------------------------------

    rainfall_candidates = [
        item for item in candidates
        if "rainfall" in item["matched_aspects"]
    ]

    if rainfall_candidates:

        item = rainfall_candidates[0].copy()
        item["selected_for"] = "rainfall"

        selected.append(item)
        selected_ids.add(item["chunk_id"])

    # --------------------------------------------------
    # 2. SOC
    # --------------------------------------------------

    soc_candidates = [
        item for item in candidates
        if (
            "soc" in item["matched_aspects"]
            and item["chunk_id"] not in selected_ids
        )
    ]

    if soc_candidates:

        item = soc_candidates[0].copy()
        item["selected_for"] = "soc"

        selected.append(item)
        selected_ids.add(item["chunk_id"])

    # --------------------------------------------------
    # 3. Land use / biodiversity
    # --------------------------------------------------

    management_candidates = [
        item for item in candidates
        if (
            (
                "land_use" in item["matched_aspects"]
                or
                "biodiversity" in item["matched_aspects"]
            )
            and item["chunk_id"] not in selected_ids
        )
    ]

    if management_candidates:

        item = management_candidates[0].copy()
        item["selected_for"] = "management/biodiversity"

        selected.append(item)
        selected_ids.add(item["chunk_id"])

    # --------------------------------------------------
    # Fill if fewer than 3
    # --------------------------------------------------

    for item in candidates:

        if len(selected) >= 3:
            break

        if item["chunk_id"] not in selected_ids:

            new_item = item.copy()
            new_item["selected_for"] = "general"

            selected.append(new_item)
            selected_ids.add(item["chunk_id"])

    return selected[:3]

In [49]:
results = retrieve_quality_balanced_top3(
    test_cases["GC07"]
)

for rank, item in enumerate(results, start=1):

    print("\n" + "=" * 90)
    print("RESULT", rank)
    print("=" * 90)

    print("Selected for:", item["selected_for"])
    print("Chunk:", item["chunk_id"])
    print("Source:", item["source_id"])
    print("Page:", item["page_number"])

    print(
        "Semantic distance:",
        round(item["best_distance"], 4)
    )

    print(
        "Quality penalty:",
        round(item["quality_penalty"], 4)
    )

    print(
        "Final score:",
        round(item["final_score"], 4)
    )

    print(
        "Matched aspects:",
        item["matched_aspects"]
    )

    print("\nTEXT:")
    print(item["text"])


RESULT 1
Selected for: rainfall
Chunk: SRC01_P0270_C005
Source: SRC01
Page: 270
Semantic distance: 0.2712
Quality penalty: 0.035
Final score: 0.3062
Matched aspects: ['original', 'rainfall', 'land_use', 'biodiversity']

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low areas (Table 4.3.1.1).

RESULT 2
Selected for: soc
Chunk: SRC01_P0227_C001
Source: SRC01
Page: 227
Semantic distance: 0.3162
Quality penalty: 0.0
Final score: 0.3162
Matched aspects: ['soc']

TEXT:
Threats to soil biodiversity - global and regional trends 197 CO2 CO2 CO2 C

In [50]:
quality_results = {}

for case_id in ["GC01", "GC02", "GC04", "GC06", "GC07"]:

    print("\n" + "#" * 100)
    print(case_id)
    print("QUESTION:", test_cases[case_id])
    print("#" * 100)

    results = retrieve_quality_balanced_top3(
        test_cases[case_id]
    )

    quality_results[case_id] = results

    for rank, item in enumerate(results, start=1):

        print("\n" + "=" * 80)
        print("RESULT", rank)
        print("=" * 80)

        print("Selected for:", item["selected_for"])
        print("Chunk:", item["chunk_id"])
        print("Source:", item["source_id"])
        print("Page:", item["page_number"])
        print(
            "Semantic distance:",
            round(item["best_distance"], 4)
        )
        print(
            "Quality penalty:",
            round(item["quality_penalty"], 4)
        )
        print(
            "Final score:",
            round(item["final_score"], 4)
        )
        print(
            "Matched aspects:",
            item["matched_aspects"]
        )

        print("\nTEXT:")
        print(item["text"])


####################################################################################################
GC01
QUESTION: A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?
####################################################################################################

RESULT 1
Selected for: rainfall
Chunk: SRC01_P0270_C005
Source: SRC01
Page: 270
Semantic distance: 0.3153
Quality penalty: 0.035
Final score: 0.3503
Matched aspects: ['original', 'rainfall', 'land_use', 'biodiversity']

TEXT:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil orga

In [51]:
def decompose_query(question):

    q = question.lower()

    queries = {
        "original": question
    }

    # ============================================================
    # SOC
    # ============================================================

    if (
        "soil carbon" in q
        or "soil organic carbon" in q
        or "soc" in q
    ):

        if (
            "very low" in q
            or "low soil" in q
            or "0.3%" in q
            or "0.4%" in q
        ):
            queries["soc"] = (
                question
                + " Effects of low soil organic carbon on soil health, "
                "soil biodiversity, soil organisms and ecosystem functioning."
            )

        elif (
            "good soil carbon" in q
            or "high soil carbon" in q
        ):
            queries["soc"] = (
                question
                + " Healthy soil organic carbon, soil biodiversity, "
                "soil ecosystem stability and maintenance."
            )

        else:
            queries["soc"] = (
                question
                + " Relationship between soil organic carbon, "
                "soil biodiversity and soil health."
            )

    # ============================================================
    # RAINFALL
    # ============================================================

    if (
        "rainfall" in q
        or "semi-arid" in q
        or "semi arid" in q
    ):

        if (
            "low rainfall" in q
            or "rainfall is low" in q
            or "semi-arid" in q
            or "semi arid" in q
        ):
            queries["rainfall"] = (
                question
                + " Low rainfall semi-arid dryland restoration, "
                "water limitation, soil water availability and "
                "vegetation establishment."
            )

        elif (
            "high rainfall" in q
            or "rainfall here is high" in q
        ):
            queries["rainfall"] = (
                question
                + " High rainfall non-water-limited agricultural soil "
                "restoration, vegetation and management practices."
            )

        else:
            queries["rainfall"] = (
                question
                + " Adequate rainfall and water availability for "
                "agricultural soil restoration and vegetation establishment."
            )

    # ============================================================
    # LAND USE / MANAGEMENT
    # ============================================================

    if (
        "wheat" in q
        or "corn" in q
        or "maize" in q
        or "monoculture" in q
        or "agroforestry" in q
    ):

        if "wheat" in q and "monoculture" in q:
            queries["land_use"] = (
                question
                + " Wheat monoculture agricultural management interventions: "
                "crop diversification, crop rotation, intercropping, "
                "agroforestry, organic amendments, reduced tillage, "
                "cover crops and practices that improve soil biodiversity "
                "and soil organic carbon."
            )

        elif (
            ("corn" in q or "maize" in q)
            and "monoculture" in q
        ):
            queries["land_use"] = (
                question
                + " Corn maize monoculture agricultural management interventions: "
                "crop diversification, crop rotation, intercropping, "
                "cover crops, organic amendments, reduced tillage and "
                "practices that improve soil biodiversity and "
                "soil organic carbon."
            )

        elif "agroforestry" in q:
            queries["land_use"] = (
                question
                + " Existing agroforestry systems, biodiversity, soil health, "
                "ecosystem services and sustainable maintenance."
            )

        else:
            queries["land_use"] = (
                question
                + " Agricultural land-use diversification, crop rotation, "
                "intercropping, agroforestry and sustainable soil management."
            )

    # ============================================================
    # BIODIVERSITY
    # ============================================================

    if (
        "biodiversity" in q
        or "species" in q
        or "species count" in q
        or "species diversity" in q
    ):

        if (
            "low species" in q
            or "declining" in q
        ):
            queries["biodiversity"] = (
                question
                + " Evidence-based agricultural management practices that "
                "increase or restore biodiversity, including crop diversification, "
                "intercropping, crop rotation, agroforestry, organic amendments, "
                "reduced tillage and habitat diversification."
            )

        elif (
            "high species" in q
            or "fine for now" in q
            or "stable" in q
        ):
            queries["biodiversity"] = (
                question
                + " Evidence-based agricultural management practices that "
                "maintain existing biodiversity and ecosystem stability, "
                "including crop diversification, intercropping, crop rotation, "
                "agroforestry, organic amendments, reduced tillage and "
                "habitat diversification."
            )

        else:
            queries["biodiversity"] = (
                question
                + " Evidence-based agricultural management practices that "
                "increase or maintain biodiversity, including crop diversification, "
                "intercropping, crop rotation, agroforestry, organic amendments, "
                "reduced tillage and habitat diversification."
            )

    return queries

In [52]:
for case_id in ["GC01", "GC02", "GC04", "GC06", "GC07"]:

    print("\n" + "=" * 80)
    print(case_id)
    print("=" * 80)

    queries = decompose_query(test_cases[case_id])

    for aspect, query in queries.items():
        print(f"\n{aspect.upper()}:")
        print(query)


GC01

ORIGINAL:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?

SOC:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Effects of low soil organic carbon on soil health, soil biodiversity, soil organisms and ecosystem functioning.

RAINFALL:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Low rainfall semi-arid dryland restoration, water limitation, soil water availability and vegetation establishment.

LAND_USE:
A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition? Wheat monoculture agricultural management interventions: crop diversification, crop rotation, intercropping, agroforestry, organic amendments, reduced tillage, cover c

In [53]:
quality_results = {}

for case_id in ["GC01", "GC02", "GC04", "GC06", "GC07"]:

    print("\n" + "#" * 100)
    print(case_id)
    print("QUESTION:", test_cases[case_id])
    print("#" * 100)

    results = retrieve_quality_balanced_top3(
        test_cases[case_id]
    )

    quality_results[case_id] = results

    for rank, item in enumerate(results, start=1):

        print("\n" + "=" * 80)
        print("RESULT", rank)
        print("=" * 80)

        print("Selected for:", item["selected_for"])
        print("Chunk:", item["chunk_id"])
        print("Source:", item["source_id"])
        print("Page:", item["page_number"])
        print("Final score:", round(item["final_score"], 4))
        print("Matched aspects:", item["matched_aspects"])

        print("\nTEXT:")
        print(item["text"])


####################################################################################################
GC01
QUESTION: A farm has low soil organic carbon, low rainfall, monoculture wheat, and low species counts. What intervention could improve its condition?
####################################################################################################

RESULT 1
Selected for: rainfall
Chunk: SRC01_P0233_C003
Source: SRC01
Page: 233
Final score: 0.3354
Matched aspects: ['original', 'soc', 'rainfall', 'land_use', 'biodiversity']

TEXT:
For example, conversion of natural ecosystems to agricultural lands has resulted in substantial environmental costs, including land degradation, increased emissions of greenhouse gases, decreased organic matter in soils, loss of biodiversity and alterations of biogeochemical and hydrological cycles (Balmford et al., 2005). Modern agriculture thus faces great challenges not only in terms of meeting the food, fibre and fuel demands of an ever-increasing h

In [54]:
# ============================================================
# GOLDEN CASE TOP-3 RETRIEVAL EVALUATION
# ============================================================

eval_cases = {
    "GC01": {
        "required_concepts": [
            ["soil organic carbon", "organic matter", "soil carbon"],
            ["biodiversity", "soil biodiversity"],
            ["management", "intervention", "diversification",
             "tillage", "organic amendment", "agroforestry",
             "intercropping", "crop rotation"]
        ]
    },

    "GC02": {
        "required_concepts": [
            ["soil organic carbon", "organic matter", "soil carbon"],
            ["biodiversity", "soil biodiversity"],
            ["maintain", "maintenance", "conservation",
             "sustainable", "protect"]
        ]
    },

    "GC04": {
        "required_concepts": [
            ["soil organic carbon", "organic matter", "soil carbon"],
            ["biodiversity", "soil biodiversity"],
            ["management", "reduced tillage", "residue",
             "diversification", "crop rotation",
             "intercropping", "agroforestry"]
        ]
    },

    "GC06": {
        "required_concepts": [
            ["soil organic carbon", "organic matter", "soil carbon"],
            ["biodiversity", "soil biodiversity"],
            ["monoculture", "diversification", "crop rotation",
             "intercropping", "cover crops",
             "organic amendment", "reduced tillage"]
        ]
    },

    "GC07": {
        "required_concepts": [
            ["soil organic carbon", "organic matter", "soil carbon"],
            ["biodiversity", "soil biodiversity"],
            ["rainfall", "water", "semi-arid", "dryland", "drought"],
            ["management", "conservation", "diversification",
             "intercropping", "agroforestry", "restoration"]
        ]
    }
}


def evaluate_case(case_id, results):

    combined_text = " ".join(
        item["text"] for item in results
    ).lower()

    concept_results = []

    for alternatives in eval_cases[case_id]["required_concepts"]:

        found = any(
            term.lower() in combined_text
            for term in alternatives
        )

        concept_results.append(found)

    covered = sum(concept_results)
    total = len(concept_results)

    coverage = covered / total

    # Case passes if at least 75% of its required
    # scientific concepts appear in retrieved Top-3 evidence
    passed = coverage >= 0.75

    return {
        "case_id": case_id,
        "concepts_covered": covered,
        "total_concepts": total,
        "coverage": coverage,
        "passed": passed
    }


evaluation_results = []

for case_id in eval_cases:

    results = quality_results[case_id]

    evaluation_results.append(
        evaluate_case(case_id, results)
    )


evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)


passed_cases = evaluation_df["passed"].sum()
total_cases = len(evaluation_df)

top3_success_rate = (
    passed_cases / total_cases
) * 100


print("\n" + "=" * 60)
print("GOLDEN CASE RETRIEVAL EVALUATION")
print("=" * 60)

print("Passed cases:", passed_cases, "/", total_cases)
print(f"Top-3 success rate: {top3_success_rate:.1f}%")

if top3_success_rate >= 80:
    print("TARGET ACHIEVED ✓")
else:
    print("TARGET NOT YET ACHIEVED")

,case_id,concepts_covered,total_concepts,coverage,passed
0,GC01,3,3,1.0,True
1,GC02,3,3,1.0,True
2,GC04,3,3,1.0,True
3,GC06,3,3,1.0,True
4,GC07,4,4,1.0,True



GOLDEN CASE RETRIEVAL EVALUATION
Passed cases: 5 / 5
Top-3 success rate: 100.0%
TARGET ACHIEVED ✓


In [55]:
def build_rag_context(results):

    context_parts = []

    for i, item in enumerate(results, start=1):

        context_parts.append(
            f"""
SOURCE {i}
Source ID: {item['source_id']}
Page: {item['page_number']}
Chunk ID: {item['chunk_id']}

Evidence:
{item['text']}
""".strip()
        )

    return "\n\n".join(context_parts)

In [56]:
question = test_cases["GC07"]

results = retrieve_quality_balanced_top3(question)

context = build_rag_context(results)

print(context)

SOURCE 1
Source ID: SRC01
Page: 270
Chunk ID: SRC01_P0270_C005

Evidence:
Where rainfall is higher and more reliable in the semi-arid zone, there is better vegetation cover of open low-tree grassland and a relatively healthy environment for humans and livestock. Cropping and crop–livestock systems dominate these areas and farmers commonly grow millet, sorghum, groundnut, maize and cowpeas. Threats to soil biodiversity in this ecoregion include wind and water erosion, loss of soil organic matter and soil nutrients, salinization and sodification and waterlogging in low areas (Table 4.3.1.1).

SOURCE 2
Source ID: SRC01
Page: 227
Chunk ID: SRC01_P0227_C001

Evidence:
Threats to soil biodiversity - global and regional trends 197 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 CO2 C C C C C C C C C C C Impacts on soil biodiversity Erosion and landslides drivers and effects on soils • Detachment, transport and deposition of soil particles by water or wind. • Loss of organic matter and changes in soil

In [ ]:
def build_rag_prompt(question, context):

    prompt = f"""
You are an evidence-grounded agricultural biodiversity and soil-health assistant.

Answer the user's question using ONLY the scientific evidence provided in the context below.

RULES:
1. Do not invent facts, numerical improvements, percentages, or timelines.
2. Do not make claims that are unsupported by the retrieved evidence.
3. Consider the user's stated soil organic carbon, rainfall, land use, and biodiversity conditions.
4. If important information is missing, clearly state what information is needed.
5. Give practical recommendations only when they are supported by the evidence.
6. If current conditions are already positive, do not recommend unnecessary drastic intervention.
7. Mention uncertainty when the evidence is insufficient.
8. Cite supporting evidence using [Source ID, Page X].
9. Keep the answer clear and understandable for a farmer or land manager.
10. Clearly distinguish between:
    - facts stated by the user,
    - findings reported by the scientific sources,
    - recommendations inferred from those findings.

11. Do not present a risk, threat, or condition mentioned in a source
    as if it definitely exists on the user's farm unless the user
    explicitly stated it.

12. Prefer cautious wording such as "the evidence indicates",
    "may help", "is associated with", or "is a potential risk"
    when the evidence does not establish a farm-specific fact.

13. Never claim that a specific user condition causes or is associated
    with an outcome unless the retrieved scientific context explicitly
    supports that relationship. Do not combine separate facts from the
    user and source into a new causal claim.
USER QUESTION:
{question}

SCIENTIFIC CONTEXT:
{context}

Provide the answer in this structure:

Assessment:
Briefly explain what the available conditions indicate.

Recommendation:
Give the most appropriate evidence-supported action or management approach.

Why:
Explain how the recommendation relates to soil health, biodiversity, rainfall,
or land use using the retrieved evidence.

Evidence:
List the sources used in the form [Source ID, Page X].

Limitations:
Mention any important missing information or uncertainty.
"""

    return prompt

In [77]:
question = test_cases["GC07"]

results = retrieve_quality_balanced_top3(question)

context = build_rag_context(results)

prompt = build_rag_prompt(
    question,
    context
)

print(prompt)


You are an evidence-grounded agricultural biodiversity and soil-health assistant.

Answer the user's question using ONLY the scientific evidence provided in the context below.

RULES:
1. Do not invent facts, numerical improvements, percentages, or timelines.
2. Do not make claims that are unsupported by the retrieved evidence.
3. Consider the user's stated soil organic carbon, rainfall, land use, and biodiversity conditions.
4. If important information is missing, clearly state what information is needed.
5. Give practical recommendations only when they are supported by the evidence.
6. If current conditions are already positive, do not recommend unnecessary drastic intervention.
7. Mention uncertainty when the evidence is insufficient.
8. Cite supporting evidence using [Source ID, Page X].
9. Keep the answer clear and understandable for a farmer or land manager.
10. Clearly distinguish between:
    - facts stated by the user,
    - findings reported by the scientific sources,
    - r

In [59]:
%pip install -U google-genai python-dotenv

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 1.3 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.1 MB 1.3 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.1 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.1 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 1.3 MB/s eta 0:00:03
   ----- ---------------------------------- 0.5/3.8 MB 1.3 MB/s eta 0:00:03
   -------- ----------

In [60]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("API key loaded successfully ✓")
else:
    print("API key NOT found ✗")

API key loaded successfully ✓


In [61]:
from google import genai

client = genai.Client(api_key=api_key)

print("Gemini client initialized ✓")

Gemini client initialized ✓


In [62]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with exactly: Gemini connection successful"
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini connection successful


In [65]:
import time
from google.genai import errors


def rag_answer(question):

    # 1. Retrieve evidence
    results = retrieve_quality_balanced_top3(question)

    # 2. Build context
    context = build_rag_context(results)

    # 3. Build prompt
    prompt = build_rag_prompt(question, context)

    # 4. Try Gemini with automatic retry
    max_retries = 3

    for attempt in range(max_retries):

        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )

            return response.text

        except errors.ServerError as e:

            if attempt < max_retries - 1:
                wait_time = 2 ** attempt

                print(
                    f"Gemini temporarily unavailable. "
                    f"Retrying in {wait_time}s..."
                )

                time.sleep(wait_time)

            else:
                raise e

In [66]:
question = test_cases["GC07"]

answer = rag_answer(question)

print(answer)

Gemini temporarily unavailable. Retrying in 1s...
Gemini temporarily unavailable. Retrying in 2s...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [67]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: Gemini connection successful"
)

print(response.text)

Gemini connection successful


In [68]:
import time
from google.genai import errors


def rag_answer(question):

    # Retrieve Top-3 scientific evidence
    results = retrieve_quality_balanced_top3(question)

    # Build scientific context
    context = build_rag_context(results)

    # Build grounded RAG prompt
    prompt = build_rag_prompt(question, context)

    # Fast model first, stronger Flash models as fallback
    models = [
        "gemini-3.5-flash-lite",
        "gemini-3.6-flash",
        "gemini-3.8-flash"
    ]

    last_error = None

    for model_name in models:

        try:
            print(f"Trying {model_name}...")

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            if response.text:
                print(f"Answer generated using {model_name} ✓")
                return response.text

        except errors.ServerError as e:
            last_error = e
            print(f"{model_name} temporarily unavailable.")

            time.sleep(1)

        except errors.ClientError as e:
            last_error = e
            print(f"{model_name} unavailable for this API key.")

    raise RuntimeError(
        "All Gemini models failed. Last error: "
        + str(last_error)
    )

In [78]:
question = test_cases["GC07"]

answer = rag_answer(question)

print("\nFINAL RAG ANSWER:\n")
print(answer)

Trying gemini-3.5-flash-lite...
Answer generated using gemini-3.5-flash-lite ✓

FINAL RAG ANSWER:

Assessment:
You are operating in a semi-arid region with low rainfall, a soil organic carbon level of 0.3%, and your current land use is monoculture wheat [User Question]. 

Findings reported by the scientific sources indicate that threats to soil biodiversity include erosion (water and wind), loss of soil organic matter and soil nutrients, salinization, sodification, and waterlogging [SRC01, Page 270]. Erosion can cause detachment, transport, and deposition of soil particles, loss of organic matter, changes in physical and chemical properties, elimination or displacement of upper soil layer inhabitants, loss of habitat quality, spread of pests and pathogens, and reduced soil biodiversity and functioning [SRC02, Page 227]. Additionally, management practices and production approaches promoting the conservation and sustainable use of biodiversity for food and agriculture are increasingly be